# Task 1 — Data Acquisition

**COMP5339 Data Engineering · Assignment 1 · EV Charger Data Integration & Augmentation**

---

This notebook programmatically retrieves the two source datasets the assignment is built on,
validates them, records their provenance, and profiles their quality.

| # | Dataset | Source | Retrieved via |
|---|---|---|---|
| 1 | EV charging locations in NSW (Dec 2025) | Transport for NSW | data.gov.au CKAN API |
| 2 | ASGS Edition 4 SA4 digital boundaries | Australian Bureau of Statistics | ABS download page (HTML parsing) |

### How to run

Run every cell from top to bottom (**Kernel → Restart Kernel and Run All Cells**). The first
cell installs the required packages. A complete run takes about 15 seconds and downloads
roughly 30 MB into a `data/` folder created next to this notebook.

The notebook is **safe to re-run**: files already downloaded are verified by checksum and
skipped rather than fetched again.

### Contents

1. [Setup and configuration](#setup)
2. [Shared helpers](#helpers) — HTTP, checksums, provenance manifest
3. [Source 1 — Transport for NSW EV chargers](#source1)
4. [Source 2 — ABS SA4 boundaries](#source2)
5. [Provenance manifest](#manifest)
6. [Data quality profile](#profile)
7. [SA4 boundaries in DuckDB](#duckdb)
8. [Summary and outputs](#summary)

**Task 2**

9. [Approach](#t2-approach)
10. [Setup](#t2-setup)
11. [Structural cleaning](#t2-structure)
12. [Operator names](#t2-operators)
13. [Charger type and status](#t2-type)
14. [Charger power ratings](#t2-rating)
15. [Addresses and postcodes](#t2-address)
16. [Coordinates and sites](#t2-coords)
17. [Duplicates and reconciliation](#t2-dupes)
18. [Derived rating fields](#t2-ratingfields)
19. [Cross-field consistency](#t2-consistency)
20. [Spatial integration with ASGS SA4](#t2-spatial)
21. [Validation and summary](#t2-validate)
22. [Outputs](#t2-outputs)

### Scope

Sections 1-8 cover **Task 1**; sections 9-22 cover **Task 2**. The notebook is self-contained: it has no local imports and
depends on nothing outside the packages installed in its first cell, so it can be executed on
its own to reproduce the acquisition stage in full.

The datasets Task 1 produces under `data/raw/` are the inputs to the cleaning and spatial
integration in Task 2, and the profiling in section 6 establishes the data quality issues that
Task 2 addresses. Task 2 writes its results to `data/interim/`, which Tasks 3 and 4 read in
turn, so every stage of the pipeline corresponds to one section of the accompanying report.

<a id="setup"></a>
## 1. Setup and configuration

### 1.1 Dependencies

Installing from within the notebook makes it reproducible on a fresh machine — including a
hosted environment such as Google Colab — without any prior setup. `pip` skips anything
already present, so re-running this cell is inexpensive.

In [1]:
%pip install -q "requests>=2.32" "beautifulsoup4>=4.13" "lxml>=5.3" "pandas>=2.2" "duckdb>=1.1"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import hashlib
import json
import re
import tempfile
import zipfile
from datetime import date, datetime, timezone
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pd.set_option("display.max_colwidth", 60)
print("Imports OK")

Imports OK


### 1.2 Configuration

Every path, URL and constant the notebook uses is defined here rather than scattered through
the cells below. If the ABS publishes a new ASGS edition or TfNSW a newer release, this is the
only cell that needs editing.

In [3]:
# --- Paths -----------------------------------------------------------------
# Everything is created relative to the notebook's own folder, so the project is
# portable: unzip it anywhere and it still works.
PROJECT_ROOT = Path.cwd()

DATA_DIR      = PROJECT_ROOT / "data"
RAW_DIR       = DATA_DIR / "raw"           # source files, exactly as downloaded
INTERIM_DIR   = DATA_DIR / "interim"       # intermediate outputs (Tasks 2-3)
PROCESSED_DIR = DATA_DIR / "processed"     # final outputs (Task 4)

RAW_TFNSW_DIR   = RAW_DIR / "tfnsw"
RAW_ABS_DIR     = RAW_DIR / "abs"
SA4_EXTRACT_DIR = RAW_ABS_DIR / "sa4_shapefile"

MANIFEST_PATH = RAW_DIR / "manifest.json"  # provenance record (see section 5)

for directory in (RAW_TFNSW_DIR, RAW_ABS_DIR, INTERIM_DIR, PROCESSED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# --- Source 1: Transport for NSW EV charging locations ----------------------
# The TfNSW portal requires a free login; data.gov.au mirrors the identical
# resource behind a public CKAN API, so the URL is discovered rather than fixed.
CKAN_API_BASE   = "https://data.gov.au/data/api/3/action"
CKAN_PACKAGE_ID = "nsw-2-ev-charging-locations"

# Assignment brief: "Retrieve the most-recent version from December 2025".
TARGET_RELEASE_YEAR  = 2025
TARGET_RELEASE_MONTH = 12

# TfNSW names each release `ev_YYYYMMDD.csv`; the embedded date is the only
# reliable version marker, because CKAN's `last_modified` field is null here.
TFNSW_CSV_DATE_PATTERN = re.compile(r"ev_(?P<date>\d{8})\.csv$", re.IGNORECASE)

# Used only if data.gov.au is unreachable, so the run stays reproducible.
TFNSW_FALLBACK_URL = (
    "https://opendata.transport.nsw.gov.au/data/dataset/"
    "be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/"
    "7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv"
)

# Checked against the downloaded header so a changed schema is caught here
# rather than deep inside the Task 2 cleaning code.
EXPECTED_TFNSW_COLUMNS = [
    "OBJECTID", "Station_name", "Station_address", "Operator",
    "Number_of_plugs", "Charger_Type", "Charger_rating",
    "Latitude", "Longitude", "LGANAME", "PCODE", "Source",
]

# --- Source 2: ABS ASGS Edition 4 SA4 boundaries ----------------------------
# The ABS offers no API for boundary files, so the download page is parsed.
ABS_BOUNDARY_PAGE_URL = (
    "https://www.abs.gov.au/statistics/standards/"
    "australian-statistical-geography-standard-asgs/"
    "edition-4-july-2026-june-2031/access-and-downloads/digital-boundary-files"
)

# Matching a pattern, not a literal filename, means a change of reference year
# or datum (e.g. SA4_2031_AUST_SHP_GDA2020.zip) is handled automatically.
ABS_SA4_ZIP_PATTERN = re.compile(
    r"SA4_(?P<year>\d{4})_AUST_SHP_(?P<datum>GDA\d{4})\.zip$", re.IGNORECASE
)

# A shapefile is a SET of sibling files; these four are mandatory.
REQUIRED_SHAPEFILE_SUFFIXES = {".shp", ".shx", ".dbf", ".prj"}

# --- HTTP behaviour ---------------------------------------------------------
# A descriptive User-Agent is good scraping etiquette: it identifies the client
# to the server operator instead of impersonating a browser.
USER_AGENT = (
    "COMP5339-Assignment1/1.0 (University of Sydney student project; "
    "data engineering coursework)"
)
REQUEST_TIMEOUT      = 60        # seconds to wait for a response
MAX_RETRIES          = 4         # retries for transient failures
RETRY_BACKOFF_FACTOR = 1.5       # exponential backoff: 1.5s, 3s, 6s, 12s
CHUNK_SIZE           = 1 << 16   # 64 KiB - stream large files, don't buffer them

print(f"Project root: {PROJECT_ROOT}")
print(f"Data folder:  {DATA_DIR.relative_to(PROJECT_ROOT)}/")

Project root: /Users/mac/USyd/Semester 2 2026/COMP5339 Data Engineering/Assignment 1
Data folder:  data/


<a id="helpers"></a>
## 2. Shared helpers

Downloading is the one step guaranteed to fail occasionally — networks drop, servers return
503, large files time out. Concentrating that fragility into a few well-behaved functions lets
the acquisition cells below read as though the network were reliable.

In [4]:
def log(message: str) -> None:
    """Print a timestamped progress message."""
    print(f"[{datetime.now():%H:%M:%S}] {message}")


def build_session() -> requests.Session:
    """
    Return a `requests.Session` that automatically retries transient failures.

    A Session (rather than bare `requests.get` calls) reuses the TCP connection
    and applies one retry policy to every request in the run.

    `Retry` deliberately covers only *transient* problems: connection errors and
    the 429/5xx status codes. A 404 is a genuine answer from the server, so it is
    raised immediately instead of being retried four times.
    """
    retry_policy = Retry(
        total=MAX_RETRIES,
        backoff_factor=RETRY_BACKOFF_FACTOR,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET", "HEAD"]),
        raise_on_status=False,
    )
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})
    adapter = HTTPAdapter(max_retries=retry_policy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def sha256_of_file(path: Path) -> str:
    """
    Compute a file's SHA-256 checksum, reading it in chunks rather than at once.

    The checksum is this project's evidence of reproducibility: two runs that
    produce the same digest downloaded byte-identical data. It is also how the
    notebook decides whether an already-downloaded file can be trusted.
    """
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(CHUNK_SIZE), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_file(session: requests.Session, url: str, destination: Path) -> Path:
    """
    Stream `url` into `destination` and return the path written.

    Two deliberate choices:

    1. `stream=True` writes the response to disk in 64 KiB chunks instead of
       holding it in memory. The ABS archive is ~30 MB, and the same code should
       still work if a future release is far larger.
    2. The download goes to a temporary `.part` file in the same folder and is
       renamed into place only once it completes. `Path.replace` is atomic on a
       single filesystem, so an interrupted run can never leave a half-written
       file that a later cell would happily read as if it were complete.
    """
    destination.parent.mkdir(parents=True, exist_ok=True)
    log(f"Downloading {url.rsplit('/', 1)[-1]}")

    with session.get(url, stream=True, timeout=REQUEST_TIMEOUT) as response:
        response.raise_for_status()

        handle = tempfile.NamedTemporaryFile(
            delete=False, dir=destination.parent, suffix=".part"
        )
        temp_path = Path(handle.name)
        try:
            with handle:
                for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                    if chunk:                      # skip keep-alive chunks
                        handle.write(chunk)
        except BaseException:
            temp_path.unlink(missing_ok=True)      # never leave a stray .part file
            raise

    temp_path.replace(destination)
    log(f"  saved {destination.name} ({destination.stat().st_size:,} bytes)")
    return destination

### 2.1 The provenance manifest

Every file retrieved is recorded in `data/raw/manifest.json` alongside the URL it came from,
*how* that URL was discovered, when it was fetched, its size and its SHA-256 checksum.

The manifest does two jobs:

* **Reproducibility** — a machine-readable record of exactly which version of each source a
  given run used, which is what lets someone else confirm they obtained the same bytes.
* **Caching** — re-running this notebook should not re-download a 30 MB shapefile that is
  already present and unchanged. The recorded checksum lets the notebook *verify* a cached
  file rather than merely assume it is intact.

In [5]:
def utc_now_iso() -> str:
    """Current UTC time as a timezone-aware ISO-8601 string."""
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def load_manifest() -> dict:
    """Read an existing manifest, tolerating a missing or corrupted file."""
    if not MANIFEST_PATH.exists():
        return {}
    try:
        return json.loads(MANIFEST_PATH.read_text(encoding="utf-8")).get("artefacts", {})
    except (json.JSONDecodeError, OSError) as error:
        # A damaged manifest must not stop the run; the cost is re-downloading.
        log(f"WARNING: could not read manifest ({error}); starting a new one.")
        return {}


def record_artefact(manifest, name, *, path, url, source, discovery_method, extra=None):
    """Add (or replace) the manifest entry for one downloaded file."""
    entry = {
        "source": source,
        "discovery_method": discovery_method,
        "url": url,
        # Stored relative to the project root so the manifest is portable.
        "path": str(path.relative_to(PROJECT_ROOT)),
        "bytes": path.stat().st_size,
        "sha256": sha256_of_file(path),
        "retrieved_at": utc_now_iso(),
    }
    entry.update(extra or {})
    manifest[name] = entry
    return entry


def is_unchanged(manifest, name, path: Path) -> bool:
    """
    True if `path` exists and its checksum matches what was recorded for `name`,
    i.e. the local copy can be trusted and the download skipped.
    """
    entry = manifest.get(name)
    if entry is None or not path.exists() or "sha256" not in entry:
        return False
    if sha256_of_file(path) != entry["sha256"]:
        log(f"WARNING: {path.name} does not match its recorded checksum; re-downloading.")
        return False
    return True


def save_manifest(manifest) -> Path:
    """Write the manifest, sorted for a stable, diff-friendly file."""
    MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    document = {
        "description": (
            "Provenance record of source datasets retrieved by the COMP5339 "
            "Assignment 1 acquisition notebook."
        ),
        "generated_at": utc_now_iso(),
        "artefacts": dict(sorted(manifest.items())),
    }
    MANIFEST_PATH.write_text(json.dumps(document, indent=2) + "\n", encoding="utf-8")
    log(f"Manifest written to {MANIFEST_PATH.relative_to(PROJECT_ROOT)}")
    return MANIFEST_PATH


# One session and one manifest are shared by both sources below.
session = build_session()
manifest = load_manifest()
log(f"Session ready; manifest holds {len(manifest)} existing entrie(s).")

[17:02:22] Session ready; manifest holds 2 existing entrie(s).


<a id="source1"></a>
## 3. Source 1 — Transport for NSW EV charging locations

The brief allows this file to be obtained either from the Transport for NSW open data portal
or via data.gov.au, noting that the TfNSW portal requires a free login. This notebook uses the
data.gov.au route, for a reason worth stating precisely.

data.gov.au does not host its own copy of the file. Its CKAN record is a **catalogue entry
that points at the TfNSW-hosted resource**, so the URL discovered below resolves to
`opendata.transport.nsw.gov.au` — the file is retrieved from Transport for NSW itself. What
data.gov.au provides is a *public API for discovering that URL*, avoiding the authenticated
browser session the portal's own interface expects.

That distinction matters for reproducibility. An authenticated download cannot be reproduced
by a reader who does not hold the same credentials, and embedding credentials in a submitted
notebook is not an acceptable alternative. Discovering the URL through the public catalogue
keeps the retrieval fully automated and runnable by anyone, which is what the brief requires.

### 3.1 What the CKAN API actually returns

Choosing the right resource needs care, for two reasons that are visible in the table below:

* the dataset also contains a PDF data dictionary, a **superseded 2021 CSV**, and an unrelated
  NRMA usage spreadsheet — none of which should be downloaded; and
* CKAN's `last_modified` field is **null for every resource**, so the obvious approach of
  "sort by date, take the newest" does not work.

The release date is therefore parsed out of the filename convention TfNSW uses,
`ev_YYYYMMDD.csv`.

In [6]:
def parse_release_date(url: str):
    """
    Extract the release date from a TfNSW resource URL, or None if it has none.

    TfNSW encodes the release in the filename (`.../ev_20251216.csv`). This is the
    only usable version marker, since CKAN's own `last_modified` is null for every
    resource in this dataset.
    """
    match = TFNSW_CSV_DATE_PATTERN.search(url)
    if match is None:
        return None
    try:
        return datetime.strptime(match.group("date"), "%Y%m%d").date()
    except ValueError:
        # A filename that looks like a date but isn't (e.g. ev_20251345.csv).
        return None


response = session.get(
    f"{CKAN_API_BASE}/package_show",
    params={"id": CKAN_PACKAGE_ID},
    timeout=REQUEST_TIMEOUT,
)
response.raise_for_status()
ckan_payload = response.json()

resources = pd.DataFrame(ckan_payload["result"]["resources"])
resources["filename"] = resources["url"].str.rsplit("/", n=1).str[-1]
resources["parsed_release_date"] = resources["url"].map(parse_release_date)

log(f"CKAN returned {len(resources)} resources for '{CKAN_PACKAGE_ID}'")
resources[["name", "format", "last_modified", "filename", "parsed_release_date"]]

[17:02:25] CKAN returned 4 resources for 'nsw-2-ev-charging-locations'


,name,format,last_modified,filename,parsed_release_date
0,EV Charging Locations in NSW,CSV,None,ev_20251216.csv,2025-12-16
1,EV Charging Locations documentation,PDF,None,ev-charging-locations-v2.1.pdf,None
2,EV Charging Stations in NSW - Not updated,CSV,None,electric-vehicle-charging-stations-nsw-20211207.csv,None
3,NRMA Co-Funded Charger Usage Report - Not updated,XLS,None,nrma-co-funded-charger-usage-report-october-2022.xlsx,None


Only two resources yield a parsable release date, and just one of those falls in the target
month. The selection rule below prefers the newest CSV released in **December 2025**, as the
brief requires, and falls back to the newest release of any month with a warning if TfNSW ever
withdraws it.

In [7]:
def discover_tfnsw_csv(payload: dict) -> dict:
    """
    Return the CKAN resource dictionary that should be downloaded.

    Selection rule, in order of preference:
      1. the newest dated CSV released in the target month (December 2025);
      2. otherwise the newest dated CSV of any month, with a warning.
    """
    if not payload.get("success"):
        raise RuntimeError(f"CKAN reported failure for package '{CKAN_PACKAGE_ID}'")

    # Keep only CSV resources whose filename carries a parsable release date;
    # this filters out the PDF data dictionary and the NRMA spreadsheet.
    dated = [
        (parse_release_date(r.get("url", "")), r)
        for r in payload["result"].get("resources", [])
        if parse_release_date(r.get("url", "")) is not None
    ]
    if not dated:
        raise RuntimeError(
            "No CSV matching the 'ev_YYYYMMDD.csv' convention was found - "
            "the dataset's publication format may have changed."
        )

    dated.sort(key=lambda pair: pair[0], reverse=True)
    in_target_month = [
        (d, r) for d, r in dated
        if d.year == TARGET_RELEASE_YEAR and d.month == TARGET_RELEASE_MONTH
    ]

    if in_target_month:
        release_date, resource = in_target_month[0]
        log(f"Selected the {release_date} release, as required by the brief.")
    else:
        release_date, resource = dated[0]
        log(
            f"WARNING: no release for {TARGET_RELEASE_YEAR}-{TARGET_RELEASE_MONTH:02d}; "
            f"falling back to the most recent available ({release_date})."
        )

    resource["_release_date"] = release_date.isoformat()
    return resource


# Discovery is the fragile part; the download itself is not. Falling back to the
# pinned URL keeps the notebook runnable if data.gov.au is down, and the manifest
# records that this is what happened.
try:
    selected = discover_tfnsw_csv(ckan_payload)
    tfnsw_url = selected["url"]
    tfnsw_release_date = selected["_release_date"]
    tfnsw_discovery = "data.gov.au CKAN API (action/package_show)"
except (requests.RequestException, ValueError, RuntimeError, KeyError) as error:
    log(f"ERROR: CKAN discovery failed ({error}); using the pinned fallback URL.")
    tfnsw_url = TFNSW_FALLBACK_URL
    tfnsw_release_date = "2025-12-16"
    tfnsw_discovery = "pinned fallback URL (CKAN discovery unavailable)"

print(f"\nURL:      {tfnsw_url}")
print(f"Released: {tfnsw_release_date}")

[17:02:25] Selected the 2025-12-16 release, as required by the brief.

URL:      https://opendata.transport.nsw.gov.au/data/dataset/be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv
Released: 2025-12-16


### 3.2 Download and validate

The downloaded file is checked against the twelve expected columns before anything else uses
it. That validation matters more than usual here, because **the server misreports the file
type**: TfNSW returns the Content-Type of an Excel workbook
(`application/vnd.openxmlformats-officedocument.spreadsheetml.sheet`) for a file whose body is
plain UTF-8 CSV. Trusting the header would hand the file to the wrong parser, so the content
itself is inspected instead.

In [8]:
def validate_ev_csv(path: Path) -> int:
    """
    Confirm the file really is the expected CSV, and return its row count.

    Read with `utf-8-sig`, not `utf-8`: the file begins with a UTF-8 byte-order
    mark, and reading it as plain utf-8 leaves a stray BOM character glued to the
    first column name - which silently breaks every later reference to it.
    """
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle)
        try:
            header = next(reader)
        except StopIteration as error:
            raise ValueError(f"{path.name} is empty") from error
        row_count = sum(1 for _ in reader)

    missing = [c for c in EXPECTED_TFNSW_COLUMNS if c not in header]
    if missing:
        raise ValueError(f"{path.name} is missing column(s): {', '.join(missing)}")

    unexpected = [c for c in header if c not in EXPECTED_TFNSW_COLUMNS]
    if unexpected:
        # Not fatal, but reported so the change is handled deliberately in Task 2.
        log(f"WARNING: unexpected column(s) in {path.name}: {', '.join(unexpected)}")
    if row_count == 0:
        raise ValueError(f"{path.name} has a header but no data rows")

    log(f"Validated {path.name}: {len(header)} columns, {row_count:,} data rows.")
    return row_count


EV_CSV_PATH = RAW_TFNSW_DIR / Path(tfnsw_url).name

if is_unchanged(manifest, "tfnsw_ev_charging_locations", EV_CSV_PATH):
    log(f"{EV_CSV_PATH.name} already present and unchanged; skipping download.")
else:
    download_file(session, tfnsw_url, EV_CSV_PATH)

ev_row_count = validate_ev_csv(EV_CSV_PATH)

record_artefact(
    manifest,
    "tfnsw_ev_charging_locations",
    path=EV_CSV_PATH,
    url=tfnsw_url,
    source=(
        "Transport for NSW - EV Charging Locations in NSW "
        "(mirrored on data.gov.au, dataset id 'nsw-2-ev-charging-locations')"
    ),
    discovery_method=tfnsw_discovery,
    extra={
        "release_date": tfnsw_release_date,
        "format": "CSV (UTF-8 with BOM)",
        "data_rows": ev_row_count,
        "note": (
            "Server reports an Excel Content-Type but serves CSV; read with "
            "encoding='utf-8-sig'."
        ),
    },
)
print(f"\n-> {EV_CSV_PATH.relative_to(PROJECT_ROOT)}")

[17:02:25] ev_20251216.csv already present and unchanged; skipping download.
[17:02:25] Validated ev_20251216.csv: 12 columns, 1,958 data rows.

-> data/raw/tfnsw/ev_20251216.csv


<a id="source2"></a>
## 4. Source 2 — ABS ASGS Edition 4 SA4 digital boundaries

The ABS publishes its boundary files as ZIP archives linked from an ordinary HTML page — there
is no API and no CKAN mirror. The page is therefore fetched and parsed with BeautifulSoup.

Links are matched against the ABS **filename convention** `SA4_<year>_AUST_SHP_<datum>.zip`
rather than one literal filename, so a change of reference year or datum is picked up
automatically and the year actually retrieved is written to the manifest.

In [9]:
def discover_sa4_zip_url(session: requests.Session) -> str:
    """
    Scrape the ABS boundary-files page and return the SA4 archive URL.

    Every download on that page is a plain `<a href="...zip">`, so no browser or
    JavaScript execution is needed. If the ABS ever listed more than one SA4
    archive, the most recent reference year wins.
    """
    page = session.get(ABS_BOUNDARY_PAGE_URL, timeout=REQUEST_TIMEOUT)
    page.raise_for_status()
    soup = BeautifulSoup(page.text, "lxml")

    matches = []
    for anchor in soup.find_all("a", href=True):
        match = ABS_SA4_ZIP_PATTERN.search(anchor["href"])
        if match:
            # Links are site-relative, so resolve them against the page URL
            # rather than concatenating strings.
            matches.append((int(match.group("year")),
                            urljoin(ABS_BOUNDARY_PAGE_URL, anchor["href"])))

    if not matches:
        raise RuntimeError(
            "No SA4 shapefile link found on the ABS page - its structure or "
            "filename convention may have changed."
        )

    matches.sort(key=lambda pair: pair[0], reverse=True)
    year, url = matches[0]
    log(f"Found the SA4 {year} boundary archive.")
    return url


sa4_url = discover_sa4_zip_url(session)
print(sa4_url)

[17:02:28] Found the SA4 2026 boundary archive.
https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-4-july-2026-june-2031/access-and-downloads/digital-boundary-files/SA4_2026_AUST_SHP_GDA2020.zip


The archive is unpacked with **each member's path checked first**. A ZIP entry is free to
contain `..` or an absolute path, which `extractall` would follow and write *outside* the
target folder (the "zip slip" problem). Rejecting those names keeps extraction contained.

Afterwards the four mandatory shapefile components — `.shp`, `.shx`, `.dbf`, `.prj` — are
confirmed present, since a shapefile is a *set* of sibling files rather than one file.

In [10]:
def extract_shapefile(archive_path: Path, target_dir: Path) -> Path:
    """Extract the archive safely and return the path of the .shp inside it."""
    if not zipfile.is_zipfile(archive_path):
        raise ValueError(
            f"{archive_path.name} is not a valid ZIP - the download may have "
            "returned an error page instead of the file."
        )
    target_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive_path) as archive:
        members = archive.namelist()
        for member in members:
            member_path = Path(member)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Refusing to extract unsafe path: {member!r}")
        archive.extractall(target_dir)

    log(f"Extracted {len(members)} file(s) to {target_dir.relative_to(PROJECT_ROOT)}")

    shp_files = sorted(target_dir.rglob("*.shp"))
    if not shp_files:
        raise ValueError(f"No .shp file found inside {archive_path.name}")

    shapefile = shp_files[0]
    present = {p.suffix.lower() for p in shapefile.parent.glob(f"{shapefile.stem}.*")}
    missing = REQUIRED_SHAPEFILE_SUFFIXES - present
    if missing:
        raise ValueError(f"Incomplete shapefile; missing {', '.join(sorted(missing))}")

    log(f"Shapefile components present: {', '.join(sorted(present))}")
    return shapefile


def read_projection(shapefile: Path) -> str:
    """
    Return the CRS name from the shapefile's `.prj` sidecar file.

    Worth recording at acquisition time, because the two datasets do NOT share a
    coordinate reference system: the ABS boundaries are GDA2020 while the TfNSW
    latitude/longitude columns are WGS84. Task 2's spatial join must reconcile
    them, and this is the evidence of what was actually supplied.
    """
    prj_text = shapefile.with_suffix(".prj").read_text(encoding="utf-8", errors="replace")
    # A .prj holds one WKT string, e.g. GEOGCS["GDA2020", ...
    first_quote = prj_text.find('"')
    second_quote = prj_text.find('"', first_quote + 1)
    return prj_text[first_quote + 1:second_quote] if first_quote != -1 else "unknown"


SA4_ZIP_PATH = RAW_ABS_DIR / Path(sa4_url).name

if is_unchanged(manifest, "abs_asgs_sa4_boundaries", SA4_ZIP_PATH):
    log(f"{SA4_ZIP_PATH.name} already present and unchanged; skipping download.")
else:
    download_file(session, sa4_url, SA4_ZIP_PATH)

# Re-extract whenever the extracted folder is absent; this also repairs a run
# interrupted between downloading and unpacking.
already_extracted = sorted(SA4_EXTRACT_DIR.rglob("*.shp"))
SA4_SHAPEFILE_PATH = (
    already_extracted[0] if already_extracted
    else extract_shapefile(SA4_ZIP_PATH, SA4_EXTRACT_DIR)
)

sa4_crs = read_projection(SA4_SHAPEFILE_PATH)
log(f"Shapefile CRS: {sa4_crs}")

record_artefact(
    manifest,
    "abs_asgs_sa4_boundaries",
    path=SA4_ZIP_PATH,
    url=sa4_url,
    source=(
        "Australian Bureau of Statistics - Australian Statistical Geography "
        "Standard (ASGS) Edition 4, July 2026 - June 2031, SA4 digital boundaries"
    ),
    discovery_method=(
        "HTML scrape of the ABS digital boundary files page (BeautifulSoup link matching)"
    ),
    extra={
        "format": "ESRI Shapefile (ZIP archive)",
        "extracted_to": str(SA4_EXTRACT_DIR.relative_to(PROJECT_ROOT)),
        "shapefile": str(SA4_SHAPEFILE_PATH.relative_to(PROJECT_ROOT)),
        "crs": sa4_crs,
    },
)
print(f"\n-> {SA4_SHAPEFILE_PATH.relative_to(PROJECT_ROOT)}")

[17:02:28] SA4_2026_AUST_SHP_GDA2020.zip already present and unchanged; skipping download.
[17:02:28] Shapefile CRS: GDA2020

-> data/raw/abs/sa4_shapefile/SA4_2026_AUST_GDA2020.shp


<a id="manifest"></a>
## 5. Provenance manifest

Both files are now recorded. Re-running this notebook re-hashes what is on disk and skips the
downloads only if the checksums still match, so a corrupted local copy is repaired rather than
trusted.

In [11]:
save_manifest(manifest)

pd.DataFrame(manifest).T[
    ["source", "discovery_method", "bytes", "sha256", "retrieved_at"]
]

[17:02:28] Manifest written to data/raw/manifest.json


,source,discovery_method,bytes,sha256,retrieved_at
abs_asgs_sa4_boundaries,Australian Bureau of Statistics - Australian Statistical...,HTML scrape of the ABS digital boundary files page (Beau...,29543412,d2df15ee57dac089457fd6ff27e342d4f6286ab0dbc587fff20cf056...,2026-09-07T07:02:28+00:00
tfnsw_ev_charging_locations,Transport for NSW - EV Charging Locations in NSW (mirror...,data.gov.au CKAN API (action/package_show),283033,43970e7751b951ab459a1be7bfd6141756c60ff8aa797debf11c5a59...,2026-09-07T07:02:25+00:00


<a id="profile"></a>
## 6. Data quality profile

The remainder of the notebook profiles what was retrieved. These figures are the evidence for
the report's *Data Cleaning and Quality Assessment* section, and they determine the cleaning
strategy that Task 2 implements.

In [12]:
ev = pd.read_csv(EV_CSV_PATH, encoding="utf-8-sig")   # utf-8-sig strips the BOM
print(f"{ev.shape[0]:,} rows x {ev.shape[1]} columns")
ev.head()

1,958 rows x 12 columns


,OBJECTID,Station_name,Station_address,Operator,Number_of_plugs,Charger_Type,Charger_rating,Latitude,Longitude,LGANAME,PCODE,Source
0,NaN,NaN,", Muswellbrook, 2333",EVUp,2,AC,22 kW,-32.262242,150.890139,Muswellbrook Shire Council,2333,Existing Destination Chargers
1,NaN,NaN,"01 Wallgrove Road, Sydney, 2766",BP,4,DC,150 kW,-33.811004,150.849597,Blacktown City Council,2766,Existing Fast Chargers
2,NaN,NaN,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",NRMA,4,DC,50 kW,-30.511874,151.669395,Central Darling Shire Council,2350,TfNSW Regional
3,NaN,NaN,"1 Balfour St, Sydney, 2070",Chargefox,7,AC,22 kW,-33.774101,151.167035,Ku-ring-gai Council,2070,Existing Destination Chargers
4,NaN,NaN,"1 Bay Ln, Byron Bay, 2481",Tesla,2,AC,19 kW,-28.641819,153.613633,Byron Shire Council,2481,Existing Destination Chargers


### 6.1 Missing values

`Station_name` is absent from most records. `LGANAME`, `PCODE` and `Source` are null in
**the same 121 rows**, which suggests those records entered the dataset through a different
process from the rest.

In [13]:
profile = pd.DataFrame({
    "dtype": ev.dtypes.astype(str),
    "nulls": ev.isna().sum(),
    "null_pct": (ev.isna().mean() * 100).round(1),
    "distinct": ev.nunique(),
})
profile

,dtype,nulls,null_pct,distinct
OBJECTID,float64,1837,93.8,121
Station_name,object,1438,73.4,477
Station_address,object,0,0.0,1924
Operator,object,0,0.0,50
Number_of_plugs,int64,0,0.0,18
Charger_Type,object,0,0.0,3
Charger_rating,object,0,0.0,46
Latitude,float64,0,0.0,1937
Longitude,float64,0,0.0,1935
LGANAME,object,121,6.2,129


### 6.2 The finding that shapes Task 3

`Charger_Type` carries a third value, `Upcoming`, which describes a charger's *status* rather
than its type — Task 2 has to decide explicitly whether those records are in scope.

More consequentially: of the **433 DC fast chargers** that Task 3 must augment, **only one has
a station name**. Name-based matching against an external source (Open Charge Map, operator
websites) is therefore impossible, which makes **coordinates the only reliable join key**.

In [14]:
print(ev["Charger_Type"].value_counts().to_string(), "\n")

name_coverage = ev.groupby("Charger_Type")["Station_name"].agg(
    records="size", with_name="count"
)
name_coverage["with_name_pct"] = (
    name_coverage["with_name"] / name_coverage["records"] * 100
).round(1)
name_coverage

Charger_Type
AC          1427
DC           433
Upcoming      98 



,records,with_name,with_name_pct
Charger_Type,,,
AC,1427,515,36.1
DC,433,1,0.2
Upcoming,98,4,4.1


### 6.3 Inconsistent operator naming

50 distinct operator strings stand for roughly 35 real operators. Three separate problems are
tangled together in this one column:

1. **Case and punctuation variants** that a normalisation function can merge on its own —
   `ChargeHub`/`Charge Hub`, `Non-networked`/`Non-Networked`.
2. **Trailing whitespace** — `'BP Australia '` and `'Tesla Motors '` both carry a trailing
   space, so they are distinct strings from their trimmed forms even before any other
   difference is considered.
3. **Truncation at 13–14 characters**, which the output below effectively proves: the dataset
   contains *both* `Viva Energy A` and `Viva Energy Australia`, and both `PLUS ES` and
   `PLUS ES Manag`. A fixed-width field somewhere upstream is cutting names short. These
   cannot be repaired by normalisation and need a manual lookup table.

A caution on the heuristic below: grouping by shared first word is a *lead generator*, not an
answer. It correctly pairs `Evie`/`Evie Networks`, but it also pairs `Charge Hub` with
`Charge OS`, which are unrelated companies. Every proposed merge needs human review before
Task 2 applies it — which is exactly why the mapping belongs in an explicit lookup table
rather than in an automatic rule.

In [15]:
print(f"{ev['Operator'].nunique()} distinct operator values\n")

# Group operators by a crude normalised key to surface likely duplicate entities.
normalised_key = ev["Operator"].str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
variant_groups = (
    pd.DataFrame({"operator": ev["Operator"], "key": normalised_key})
      .drop_duplicates()
      .groupby("key")["operator"]
      .apply(list)
)
print("Variants differing only by case or punctuation:")
for names in variant_groups[variant_groups.map(len) > 1]:
    print("   ", names)

# Operators sharing a leading word are likely the same entity written two ways.
print("\nOperators sharing a first word (possible same entity):")
first_word = pd.Series(ev["Operator"].unique()).str.split().str[0].str.lower()
for word, group in pd.Series(ev["Operator"].unique()).groupby(first_word):
    if len(group) > 1:
        print("   ", sorted(group))

# Trailing/leading whitespace makes otherwise identical names distinct strings.
whitespace_affected = [n for n in ev["Operator"].unique() if n != n.strip()]
print("\nOperator values with leading/trailing whitespace:", whitespace_affected)

# Where one operator name is a strict prefix of another, the pair is either a
# naming variant ('BP' / 'BP Australia') or genuine truncation
# ('Viva Energy A' / 'Viva Energy Australia'). The test cannot tell them apart,
# so it reports the candidates and flags which ones end mid-word - a name cut
# off at a partial word is the signature of truncation rather than abbreviation.
unique_operators = sorted(ev["Operator"].unique(), key=len)
prefix_pairs = [
    (short, longer)
    for short in unique_operators
    for longer in unique_operators
    if short != longer and longer.startswith(short.rstrip()) and len(short.strip()) <= 20
]
print("\nPrefix pairs (naming variant or truncation - needs review):")
for short, longer in prefix_pairs:
    # If the longer name continues the short one mid-word, the short form was cut.
    cut_mid_word = len(longer) > len(short.rstrip()) and longer[len(short.rstrip())] != " "
    marker = "  <- truncated mid-word" if cut_mid_word else ""
    print(f"    {short!r:<24} -> {longer!r}{marker}")

print()
ev["Operator"].value_counts().head(15)

50 distinct operator values

Variants differing only by case or punctuation:
    ['ChargeHub', 'Charge Hub']
    ['Non-networked', 'Non-Networked']

Operators sharing a first word (possible same entity):
    ['BP', 'BP Australia ']
    ['Charge Hub', 'Charge OS']
    ['Evie', 'Evie Networks']
    ['Non-Networked', 'Non-networked']
    ['NRMA', 'NRMA Electric']
    ['PLUS ES', 'PLUS ES Manag']
    ['Porsche Destination Charging', 'Porsche Smart Mobility']
    ['Tesla', 'Tesla Motors ']
    ['Viva Energy A', 'Viva Energy Australia']

Operator values with leading/trailing whitespace: ['BP Australia ', 'Tesla Motors ']

Prefix pairs (naming variant or truncation - needs review):
    'BP'                     -> 'BP Australia '
    'NRMA'                   -> 'NRMA Electric'
    'Evie'                   -> 'Evie Networks'
    'Tesla'                  -> 'Tesla Motors '
    'PLUS ES'                -> 'PLUS ES Manag'
    'Viva Energy A'          -> 'Viva Energy Australia'  <- truncated mid-wo

Operator
Exploren         301
Tesla            260
Chargefox        254
Non-networked    217
PLUS ES          150
EVX              111
Evie              84
NRMA              78
JOLT              49
EVUp              38
Everty            36
Ampol             32
BP                32
BP Australia      28
Tesla Motors      27
Name: count, dtype: int64

### 6.4 Inconsistent charger attributes

`Charger_rating` mixes at least four formats in a single column: a value with units (`22 kW`),
a bare number (`22`), a non-numeric placeholder (`AC`, in 522 rows), and compound
multi-connector descriptions (`2x350kW & 2x175kW`).

Turning this into something queryable is a genuine **schema design decision** for Task 4 — not
just a string cleanup — because a compound value describes several connectors at one site.

In [16]:
print(f"{ev['Charger_rating'].nunique()} distinct rating strings\n")

def rating_format(value) -> str:
    """Classify a raw Charger_rating string by the format it uses."""
    value = str(value).strip()
    if re.fullmatch(r"\d+(\.\d+)?\s*kW", value, re.IGNORECASE):
        return "number + unit  (e.g. '22 kW')"
    if re.fullmatch(r"\d+(\.\d+)?", value):
        return "bare number    (e.g. '22')"
    if re.search(r"\dx\s*\d", value, re.IGNORECASE):
        return "compound       (e.g. '2x350kW & 2x175kW')"
    if re.fullmatch(r"[A-Za-z ]+", value):
        return "non-numeric    (e.g. 'AC')"
    return "other"

print(ev["Charger_rating"].map(rating_format).value_counts().to_string())
print()
ev["Charger_rating"].value_counts().head(12)

46 distinct rating strings

Charger_rating
number + unit  (e.g. '22 kW')                1315
non-numeric    (e.g. 'AC')                    522
compound       (e.g. '2x350kW & 2x175kW')      99
bare number    (e.g. '22')                     22



Charger_rating
22 kW                634
AC                   522
6 kW                 112
50 kW                 86
2x350kW & 2x175kW     85
75 kW                 80
7 kW                  66
25 kW                 52
175 kW                46
150 kW                44
11 kW                 31
125 kW                19
Name: count, dtype: int64

### 6.5 Duplicates and coordinate sanity

There are no exactly duplicated rows, but some coordinate pairs recur. These must be
*examined* rather than blindly dropped: one physical site can legitimately host separate AC and
DC units, which are genuinely different records.

Every coordinate falls inside the NSW bounding box, so the geometry itself is sound and the
spatial join in Task 2 should not encounter stray points.

In [17]:
print("Exact duplicate rows:                 ", ev.duplicated().sum())
print("Duplicate (lat, lon):                 ",
      ev.duplicated(subset=["Latitude", "Longitude"]).sum())
print("Duplicate (lat, lon, operator, type): ",
      ev.duplicated(subset=["Latitude", "Longitude", "Operator", "Charger_Type"]).sum())

NSW_LAT = (-37.6, -28.1)      # approximate NSW bounding box
NSW_LON = (140.9, 153.7)
outside = ev[~ev["Latitude"].between(*NSW_LAT) | ~ev["Longitude"].between(*NSW_LON)]

print(f"\nLatitude range:  {ev.Latitude.min():.4f} to {ev.Latitude.max():.4f}")
print(f"Longitude range: {ev.Longitude.min():.4f} to {ev.Longitude.max():.4f}")
print(f"Points outside the NSW bounding box: {len(outside)}")

# The duplicated coordinates, shown so Task 2 can decide case by case.
ev[ev.duplicated(subset=["Latitude", "Longitude"], keep=False)].sort_values(
    ["Latitude", "Longitude"]
)[["Station_address", "Operator", "Charger_Type", "Charger_rating", "Number_of_plugs"]].head(10)

Exact duplicate rows:                  0
Duplicate (lat, lon):                  20
Duplicate (lat, lon, operator, type):  11

Latitude range:  -37.1115 to -28.1686
Longitude range: 141.4601 to 153.6159
Points outside the NSW bounding box: 0


,Station_address,Operator,Charger_Type,Charger_rating,Number_of_plugs
604,520 David St\nAlbury NSW 2640\nAustralia,Exploren,AC,AC,4
1087,"520 David St, Albury NSW 2640, Australia",Exploren,AC,7,2
530,406 Moppett St\nHay NSW 2711\nAustralia,Exploren,AC,AC,4
1073,"406 Moppett St, Hay NSW 2711, Australia",Exploren,AC,22,2
245,"19 Princes Hwy, Figtree NSW 2525, Australia",Tesla,DC,175 kW,6
1088,"19 Princes Hwy, Figtree NSW 2525, Australia",Tesla Motors,DC,2x350kW & 2x175kW,6
1150,"19 Princes Hwy, Figtree NSW 2525",Non-networked,AC,AC,2
967,"University of Wollongong, Northfields Avenue, Wollongong...",Chargefox,DC,175 kW,6
1061,"University of Wollongong, Northfields Avenue, Wollongong...",University of,DC,2x350kW & 2x175kW,6
1775,Early Start Discovery Space (Building 21) University of ...,Chargefox,AC,AC,4


### 6.6 Address formats

`Station_address` appears in at least three formats, and **733 of the 1,958 records contain
embedded newlines** — enough that any naive string comparison against an external source will
fail on more than a third of the data.

The locality component is separately unreliable: **399 records** give the locality as the
placeholder `Sydney` rather than the true suburb. `293 Belmore Rd, Sydney, 2210` is in
Riverwood, some 15 km from the CBD.

Address matching in Task 3 therefore cannot depend on the locality field; the postcode and the
street line are the trustworthy parts.

In [18]:
for address in ev["Station_address"].sample(6, random_state=1):
    print(repr(address))

print("\nRecords whose locality is the placeholder 'Sydney':",
      ev["Station_address"].str.contains(r",\s*Sydney\s*,", regex=True, na=False).sum())
print("Records containing an embedded newline:            ",
      ev["Station_address"].str.contains("\n", na=False).sum())

'McFarlane St, Sydney, 2160'
'697 Wollombi Rd\nBroke NSW 2330\nAustralia'
'154 Beach Rd, Batemans Bay NSW 2536'
'53 Macquarie Rd\nCardiff NSW 2285\nAustralia'
'76 Wingewarra St, Dubbo , 2830'
'Lauder St, Tumbarumba, 2653'

Records whose locality is the placeholder 'Sydney': 399
Records containing an embedded newline:             733


<a id="duckdb"></a>
## 7. SA4 boundaries in DuckDB

The shapefile is read with DuckDB's `spatial` extension rather than GeoPandas. Two reasons:

* it keeps the dependency list small (GeoPandas pulls in GDAL, Fiona and Shapely); and
* more importantly, the point-in-polygon join in Task 2 and the final storage in Task 4 then
  happen in the **same engine the assignment requires**, instead of moving data between two.

In [19]:
import duckdb

con = duckdb.connect()                       # in-memory; Task 4 will use a file
con.execute("INSTALL spatial; LOAD spatial;")

con.execute(
    f"CREATE OR REPLACE VIEW sa4 AS SELECT * FROM ST_Read('{SA4_SHAPEFILE_PATH.as_posix()}')"
)
con.execute("DESCRIBE sa4").df()

,column_name,column_type,null,key,default,extra
0,OGC_FID,BIGINT,YES,None,None,None
1,SA4_CODE26,VARCHAR,YES,None,None,None
2,SA4_NAME26,VARCHAR,YES,None,None,None
3,CHG_FLAG26,VARCHAR,YES,None,None,None
4,CHG_LBL26,VARCHAR,YES,None,None,None
5,GCC_CODE26,VARCHAR,YES,None,None,None
6,GCC_NAME26,VARCHAR,YES,None,None,None
7,STE_CODE26,VARCHAR,YES,None,None,None
8,STE_NAME26,VARCHAR,YES,None,None,None
9,AUS_CODE26,VARCHAR,YES,None,None,None


The geometry column reports **EPSG:7844** (GDA2020), whereas the TfNSW latitude/longitude
columns are WGS84. The two datasets do not share a coordinate reference system — something
Task 2's spatial join has to reconcile explicitly rather than assume away.

The ABS publishes SA4 boundaries for the whole of Australia, so NSW is filtered in Task 2:
**30 of the 108 SA4 regions**.

In [20]:
con.execute('''
    SELECT STE_NAME26 AS state, COUNT(*) AS sa4_regions
    FROM sa4
    GROUP BY state
    ORDER BY sa4_regions DESC
''').df()

,state,sa4_regions
0,New South Wales,30
1,Queensland,21
2,Victoria,19
3,Western Australia,12
4,South Australia,9
5,Tasmania,6
6,Northern Territory,4
7,Australian Capital Territory,3
8,Other Territories,3
9,Outside Australia,1


In [21]:
con.execute('''
    SELECT SA4_CODE26, SA4_NAME26, GCC_NAME26, ROUND(AREASQKM26, 1) AS area_sqkm
    FROM sa4
    WHERE STE_NAME26 = 'New South Wales'
    ORDER BY SA4_CODE26
''').df()

,SA4_CODE26,SA4_NAME26,GCC_NAME26,area_sqkm
0,101,Capital Region,Rest of NSW,51896.2
1,102,Central Coast,Greater Sydney,1681.0
2,103,Central West,Rest of NSW,70297.1
3,104,Coffs Harbour - Grafton,Rest of NSW,13229.8
4,105,Far West and Orana,Rest of NSW,339355.6
5,106,Hunter Valley exc Newcastle,Rest of NSW,21491.3
6,107,Illawarra,Rest of NSW,1539.2
7,108,Mid North Coast,Rest of NSW,18851.5
8,109,Murray,Rest of NSW,97796.5
9,110,New England and North West,Rest of NSW,99139.9


In [22]:
con.close()
session.close()
log("Task 1 complete.")

[17:02:28] Task 1 complete.


<a id="summary"></a>
## 8. Summary and outputs

### What was retrieved

| Dataset | File | Size | Records |
|---|---|---|---|
| TfNSW EV charging locations (16 Dec 2025) | `data/raw/tfnsw/ev_20251216.csv` | 283 KB | 1,958 chargers |
| ABS ASGS Ed. 4 SA4 boundaries (2026, GDA2020) | `data/raw/abs/SA4_2026_AUST_SHP_GDA2020.zip` | 29.5 MB | 108 SA4 regions (30 in NSW) |

Both URLs were **discovered at run time** — via the CKAN API and by parsing the ABS page —
rather than hard-coded, so the notebook keeps working when the publishers issue a new release.
Nothing was downloaded by hand.

### Reproducibility features

* **Retries with exponential backoff** on connection errors and 429/5xx responses; a 404 is a
  real answer and is raised rather than retried.
* **Atomic downloads** — each file streams to a temporary `.part` and is renamed into place
  only on success, so an interrupted run cannot leave a truncated file behind.
* **Checksum-verified caching** — a re-run re-hashes what is on disk and skips the download
  only if it matches the manifest.
* **Content validation** — the CSV header is checked against the twelve expected columns and
  its row count confirmed non-zero; the ZIP is verified as a real archive containing a complete
  shapefile.
* **Safe extraction** — archive members with `..` or absolute paths are rejected.

### Data quality issues identified

| # | Issue | Detail |
|---|---|---|
| 1 | Missing station names | Null in 1,438 records — **including 432 of 433 DC chargers** |
| 2 | Inconsistent operator naming | 50 values for ~35 entities; case/punctuation variants, trailing whitespace, and names truncated at 13–14 chars |
| 3 | Inconsistent charger attributes | `Charger_rating` mixes 4 formats across 46 distinct strings (1,315 unit-suffixed, 522 non-numeric, 99 compound, 22 bare) |
| 4 | Status mixed into type | `Charger_Type` includes `Upcoming` (98 rows) |
| 5 | Correlated missingness | `LGANAME`, `PCODE`, `Source` null in the same 121 rows |
| 6 | Duplicate coordinates | 20 records share a location with another record |
| 7 | Address inconsistency | 3 formats; 733 records contain embedded newlines; 399 use `Sydney` as a placeholder locality |
| 8 | CRS mismatch | ABS boundaries are GDA2020 (EPSG:7844); TfNSW columns are WGS84 |

### Outputs of this stage

Running this notebook produces the following, all under `data/raw/`. These are the inputs to
Task 2, which reads them directly rather than repeating the retrieval.

| File | Contents |
|---|---|
| `tfnsw/ev_20251216.csv` | 1,958 raw EV charger records, unmodified as downloaded |
| `abs/SA4_2026_AUST_SHP_GDA2020.zip` | ABS SA4 boundary archive, unmodified as downloaded |
| `abs/sa4_shapefile/` | The extracted shapefile: 108 SA4 boundary polygons |
| `manifest.json` | Provenance record — source URL, discovery method, retrieval timestamp, byte size and SHA-256 checksum for each file |

Within this notebook these are held in `EV_CSV_PATH`, `SA4_SHAPEFILE_PATH` and
`MANIFEST_PATH`.

Source files are written to `data/raw/` and never modified in place; cleaned and derived data
are written to `data/interim/` and `data/processed/` by later stages. Keeping the raw layer
immutable means any result can be reproduced from the downloaded files alone, and that a bug
in a later stage cannot corrupt the evidence it was derived from.

### Implications for the remaining tasks

Three findings above constrain how the later stages must be approached:

* **Augmentation cannot use station names.** With 432 of 433 DC chargers unnamed, matching to
  an external source such as Open Charge Map has to be driven by coordinates, with address
  fields serving only as a secondary check.
* **Address text is not a reliable matching key either.** 733 records contain embedded
  newlines and 399 carry a placeholder locality, so only the street line and postcode can be
  trusted.
* **The spatial join must handle two coordinate reference systems.** The ABS boundaries are
  published in GDA2020 (EPSG:7844) while the TfNSW coordinates are WGS84, so the join has to
  reconcile them explicitly rather than assume they align.

---

# Task 2 — Data Integration and Cleaning

<a id="t2-approach"></a>
## 9. Approach

Task 1 established *what* is wrong with the source data. This half of the notebook fixes it and
joins the result to the ABS geography, producing the cleaned dataset that Tasks 3 and 4 build on.

Three principles run through everything below.

**Nothing is discarded silently.** Every transformation appends an entry to a *cleaning ledger*
recording what was done and how many records it touched. That ledger is written to
`data/interim/cleaning_log.json` and printed at the end, and it is the evidence for the report's
*Data Cleaning and Quality Assessment* section — a claim like "we removed duplicates" is worth
much less than "4 exact duplicates removed, 11 conflicting pairs reconciled under stated rules".

**Repair where a rule can be justified; flag where it cannot.** A trailing space in `'BP Australia '`
has one obviously correct fix. A locality recorded as `Sydney` for an address in Riverwood does
not — inventing the true suburb would be fabricating data. Problems of the second kind get a
boolean `*_flag` column, so a downstream query can exclude unreliable records instead of trusting
a value someone guessed. The cleaned output therefore carries eight such flags alongside the data.

**Order matters.** Whitespace is normalised before anything compares strings, deduplication runs
after operator names are canonicalised (otherwise `Tesla` and `Tesla Motors ` at the same site look
like two different chargers), and the derived power columns are computed *after* deduplication so
they always describe the record that actually survived. Each of those orderings is noted where it
occurs.

### What Task 2 produces

| File | Contents |
|---|---|
| `data/interim/ev_chargers_clean.csv` | one row per charger, cleaned, with SA4 fields and quality flags |
| `data/interim/charger_power_ratings.csv` | one row per connector group, unpacking the compound ratings |
| `data/interim/sa4_regions_nsw.csv` | the 30 NSW SA4 regions, used as the region dimension in Task 4 |
| `data/interim/cleaning_log.json` | the cleaning ledger — every step and the records it affected |

<a id="t2-setup"></a>
## 10. Setup

Task 2 reads the files Task 1 downloaded. If both halves run in one kernel it simply reuses the
paths already defined; if this section is run on its own it rebuilds them from
`data/raw/manifest.json` instead. That means the cleaning stage never silently depends on Task 1's
variables still being in memory — it depends on Task 1's *outputs*, which is the correct
relationship between two pipeline stages.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

# --- Locate the Task 1 outputs ---------------------------------------------
# If Task 1 ran in this same kernel the paths are already defined. If this half
# of the notebook is run on its own, they are rebuilt from the manifest instead,
# so Task 2 never depends on Task 1's variables still being in memory.
if "PROJECT_ROOT" not in dir():
    PROJECT_ROOT = Path.cwd()
    DATA_DIR = PROJECT_ROOT / "data"
    RAW_DIR = DATA_DIR / "raw"
    INTERIM_DIR = DATA_DIR / "interim"
    PROCESSED_DIR = DATA_DIR / "processed"
    MANIFEST_PATH = RAW_DIR / "manifest.json"

for directory in (INTERIM_DIR, PROCESSED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if "EV_CSV_PATH" not in dir() or "SA4_SHAPEFILE_PATH" not in dir():
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError(
            "data/raw/manifest.json is missing - run the Task 1 cells above first."
        )
    _artefacts = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))["artefacts"]
    EV_CSV_PATH = PROJECT_ROOT / _artefacts["tfnsw_ev_charging_locations"]["path"]
    SA4_SHAPEFILE_PATH = next((RAW_DIR / "abs" / "sa4_shapefile").rglob("*.shp"))

# --- Task 2 outputs ---------------------------------------------------------
CLEAN_CHARGERS_PATH = INTERIM_DIR / "ev_chargers_clean.csv"
RATINGS_PATH = INTERIM_DIR / "charger_power_ratings.csv"
SA4_NSW_PATH = INTERIM_DIR / "sa4_regions_nsw.csv"
CLEANING_LOG_PATH = INTERIM_DIR / "cleaning_log.json"

# --- Reference values used by the cleaning rules ----------------------------
NSW_LAT_RANGE = (-37.6, -28.1)
NSW_LON_RANGE = (140.9, 153.7)
COORDINATE_DECIMALS = 6          # ~0.11 m - finer than the source can justify
NSW_POSTCODE_RANGES = ((1000, 1999), (2000, 2599), (2619, 2899), (2921, 2999))

SOURCE_CRS = "EPSG:7844"         # GDA2020, the datum of the ABS boundaries
TARGET_CRS = "EPSG:4326"         # WGS84, the datum of the TfNSW lat/lon columns

# `cleaning_log` accumulates one entry per transformation: what was done and how
# many records it touched. It is written to disk at the end and is the evidence
# base for the report's Data Cleaning and Quality Assessment section.
cleaning_log = []


def record_step(step: str, detail: str, affected: int) -> None:
    """Append one auditable entry to the cleaning ledger and echo it."""
    cleaning_log.append({"step": step, "detail": detail, "records_affected": int(affected)})
    print(f"  {step:<26} {affected:>6,}  {detail}")


print(f"EV CSV    : {EV_CSV_PATH}")
print(f"Shapefile : {SA4_SHAPEFILE_PATH}")

EV CSV    : /Users/first/Documents/Learning_Materials/SEM2/COMP5339/Labs/assignment1/COMP5339/data/raw/tfnsw/ev_20251216.csv
Shapefile : /Users/first/Documents/Learning_Materials/SEM2/COMP5339/Labs/assignment1/COMP5339/data/raw/abs/sa4_shapefile/SA4_2026_AUST_GDA2020.shp


<a id="t2-structure"></a>
## 11. Structural cleaning

### 11.1 Reading the file as text

Section 6 read the CSV with pandas' default type inference, which was fine for profiling. For
cleaning it is not: inference is the first place data gets altered without anyone deciding that it
should be. Reading every column as text moves each conversion into a later cell where it is
explicit, ordered, and counted.

In [2]:
# `dtype=str` for every column, deliberately. pandas' type inference is the first
# place data gets silently altered: PCODE would become an integer (losing any
# leading zero and turning the 121 missing values into floats) and OBJECTID would
# become a float for the same reason. Reading everything as text means every
# conversion below is explicit, ordered, and reviewable.
raw = pd.read_csv(EV_CSV_PATH, encoding="utf-8-sig", dtype=str, keep_default_na=False)

print(f"Loaded {len(raw):,} rows x {raw.shape[1]} columns, all as text")
raw.head(3)

Loaded 1,958 rows x 12 columns, all as text


,OBJECTID,Station_name,Station_address,Operator,Number_of_plugs,Charger_Type,Charger_rating,Latitude,Longitude,LGANAME,PCODE,Source
0,,,", Muswellbrook, 2333",EVUp,2,AC,22 kW,-32.26224229,150.8901391,Muswellbrook Shire Council,2333,Existing Destination Chargers
1,,,"01 Wallgrove Road, Sydney, 2766",BP,4,DC,150 kW,-33.81100405,150.8495966,Blacktown City Council,2766,Existing Fast Chargers
2,,,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",NRMA,4,DC,50 kW,-30.5118739,151.669395,Central Darling Shire Council,2350,TfNSW Regional


### 11.2 Whitespace and missing values

This is the first substantive step because everything downstream compares strings. 733 addresses
contain embedded newlines and two operator names carry a trailing space; until those are gone, a
record cannot be matched against its own duplicate, and `'BP Australia '` is a different operator
from `'BP Australia'`.

Empty strings become proper missing values in the same pass. A `''` and a `NaN` mean the same thing
to a reader and completely different things to `isna()`, `groupby()` and a `NOT NULL` constraint,
so the ambiguity is removed once, here, rather than handled repeatedly later.

In [3]:
def squash_whitespace(value):
    """
    Collapse newlines, tabs and runs of spaces into single spaces, then trim.

    This runs before everything else because every later comparison - dedup,
    operator matching, address parsing - compares strings. 733 addresses contain
    embedded newlines and two operator names carry a trailing space, so without
    this step a large share of the dataset fails to match its own duplicate.
    """
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return pd.NA
    text = re.sub(r"\s+", " ", str(value)).strip()
    return text if text else pd.NA


df = raw.copy()
text_columns = df.columns.tolist()

blank_cells = int((df[text_columns] == "").sum().sum())
for column in text_columns:
    df[column] = df[column].map(squash_whitespace)
rewritten = int((raw[text_columns].astype(str) != df[text_columns].astype(str)).sum().sum())

print("Normalising whitespace and converting empty strings to NA")
record_step("whitespace normalised", "cells whose text was rewritten", rewritten - blank_cells)
record_step("empty -> NA", "empty strings converted to missing values", blank_cells)

Normalising whitespace and converting empty strings to NA
  whitespace normalised         808  cells whose text was rewritten
  empty -> NA                 3,638  empty strings converted to missing values


### 11.3 Column names and data types

Renaming to `snake_case` is not cosmetic. The source mixes three conventions in twelve columns
(`OBJECTID`, `Station_name`, `LGANAME`), and the destination is a SQL schema in Task 4 where a
single convention avoids quoted identifiers in every query.

The type conversions use `errors="coerce"`, which turns anything unparsable into a missing value
rather than raising. Used carelessly that hides problems, so each conversion counts how many
missing values it *created* and logs them — a coercion that silently destroys data is then
impossible to miss. `Int64` (capital I) rather than `int64` is deliberate too: plain NumPy integers
cannot represent a missing value, so `OBJECTID`, which is null in 1,837 rows, would be forced back
to float.

In [4]:
# snake_case throughout. The source mixes three conventions (`OBJECTID`,
# `Station_name`, `LGANAME`), and the destination is a SQL schema in Task 4 where
# one convention avoids quoted identifiers everywhere.
COLUMN_RENAMES = {
    "OBJECTID": "source_object_id",
    "Station_name": "station_name",
    "Station_address": "station_address_raw",
    "Operator": "operator_raw",
    "Number_of_plugs": "number_of_plugs",
    "Charger_Type": "charger_type_raw",
    "Charger_rating": "charger_rating_raw",
    "Latitude": "latitude",
    "Longitude": "longitude",
    "LGANAME": "lga_name",
    "PCODE": "postcode_reported",
    "Source": "source_feed",
}
df = df.rename(columns=COLUMN_RENAMES)

# `errors="coerce"` turns anything unparsable into NA rather than raising - and
# the number of values it silently created is logged, so a bad coercion can never
# slip past unnoticed. Int64 (capital I) is pandas' nullable integer: plain int64
# cannot hold a missing value and would force these columns back to float.
numeric_specs = {
    "source_object_id": "Int64",
    "number_of_plugs": "Int64",
    "latitude": "float64",
    "longitude": "float64",
}
for column, dtype in numeric_specs.items():
    before_missing = df[column].isna().sum()
    converted = pd.to_numeric(df[column], errors="coerce")
    df[column] = converted.astype(dtype) if dtype == "Int64" else converted
    coerced = int(df[column].isna().sum() - before_missing)
    if coerced:
        record_step("type coercion", f"{column}: unparsable values set to NA", coerced)

# Coordinates rounded to 6 dp. The source carries more decimals than a GPS fix
# justifies, and unrounded floats make two records at one physical site compare
# as different locations.
df["latitude"] = df["latitude"].round(COORDINATE_DECIMALS)
df["longitude"] = df["longitude"].round(COORDINATE_DECIMALS)

df.dtypes.to_frame("dtype")

,dtype
source_object_id,Int64
station_name,object
station_address_raw,object
operator_raw,object
number_of_plugs,Int64
charger_type_raw,object
charger_rating_raw,object
latitude,float64
longitude,float64
lga_name,object


<a id="t2-operators"></a>
## 12. Operator names

Section 6.3 showed 50 distinct operator strings standing for roughly 35 real operators, with three
distinct causes tangled together. Only one of them can be fixed by a rule.

Case and spacing variants (`ChargeHub` / `Charge Hub`) are mechanical. Short forms and full names
(`BP` / `BP Australia`) require knowing that they are the same company. Truncation at 13–14
characters (`Viva Energy A`, `PLUS ES Manag`) is caused by a fixed-width field somewhere upstream
and is *irreversible from the data alone* — the missing characters simply are not there.

So the repairs live in an explicit lookup table rather than in a similarity heuristic. A heuristic
that merges `Evie` with `Evie Networks` on a shared first word also merges `Charge Hub` with
`Charge OS`, which are unrelated companies, and nothing in the data distinguishes those two cases.
A lookup table is reviewable, arguable, and correct; a fuzzy match is none of the three.

Two decisions are recorded as deliberate non-merges: `Charge Hub` / `Charge OS`, and Porsche's two
programmes. One truncation, `University of`, cannot be resolved without an external source, so it
is flagged rather than guessed at. Any operator absent from the table is printed, so a future TfNSW
release with new operators is noticed rather than passed through unexamined.

In [5]:
# Three problems live in `Operator`, and only the first can be fixed by a rule:
# case/punctuation variants, trailing whitespace, and names truncated at 13-14
# characters by a fixed-width field upstream. Truncation is irreversible from the
# data alone, so the repairs are stated in a lookup table a human can review -
# not inferred by a string-similarity heuristic, which would also merge the
# unrelated 'Charge Hub' and 'Charge OS'.
OPERATOR_CANONICAL = {
    # case and spacing variants
    "chargehub": "ChargeHub",
    "charge hub": "ChargeHub",
    "non-networked": "Non-networked",
    "non networked": "Non-networked",
    # short form and full name of one company
    "bp": "BP Australia",
    "bp australia": "BP Australia",
    "tesla": "Tesla",
    "tesla motors": "Tesla",
    "nrma": "NRMA",
    "nrma electric": "NRMA",
    "evie": "Evie Networks",
    "evie networks": "Evie Networks",
    # cut off mid-word by the upstream fixed-width field
    "plus es": "PLUS ES",
    "plus es manag": "PLUS ES",
    "viva energy a": "Viva Energy Australia",
    "viva energy australia": "Viva Energy Australia",
}

# Deliberately NOT merged - these justifications belong in the report:
#   'Charge Hub' vs 'Charge OS'                          unrelated companies
#   'Porsche Destination Charging' vs
#   'Porsche Smart Mobility'                             two distinct programmes
# Truncations that cannot be resolved without an external source are flagged
# rather than guessed at:
UNRESOLVED_TRUNCATIONS = {"university of"}


def canonical_operator(value):
    """Map a raw operator string to its canonical form via the lookup table."""
    if pd.isna(value):
        return pd.NA
    key = re.sub(r"\s+", " ", str(value)).strip().lower()
    return OPERATOR_CANONICAL.get(key, str(value).strip())


operator_key = df["operator_raw"].fillna("").str.strip().str.lower()
df["operator"] = df["operator_raw"].map(canonical_operator)
df["operator_truncated_flag"] = operator_key.isin(UNRESOLVED_TRUNCATIONS)

print(f"Operator values: {df['operator_raw'].nunique()} raw -> {df['operator'].nunique()} canonical")
record_step("operator canonicalised", "values rewritten to a canonical name",
            int((df["operator"] != df["operator_raw"]).sum()))
record_step("operator truncated", "unresolved truncations flagged for review",
            int(df["operator_truncated_flag"].sum()))

# Anything absent from the lookup table is listed, so the table can be extended
# deliberately when TfNSW publishes a release containing new operators.
unmapped = sorted(set(df.loc[~operator_key.isin(OPERATOR_CANONICAL), "operator"].dropna()))
print(f"\n{len(unmapped)} operator(s) passed through unmapped (whitespace-normalised only):")
print("   ", ", ".join(unmapped))

Operator values: 50 raw -> 42 canonical
  operator canonicalised        181  values rewritten to a canonical name
  operator truncated              1  unresolved truncations flagged for review

34 operator(s) passed through unmapped (whitespace-normalised only):
    360 EV Charge, AXCharge, Alchemy Charge, Ampol, BMW, CasaCharge, Charge OS, ChargePoint, ChargePost, Chargefox, Chargestar, Counties Energy, EV Meter, EVE Australia, EVNet, EVSE, EVUp, EVX, Elanga, Energy Austra, Engie, Everty, Exploren, Fast Cities A, Gentari, JOLT, Noodoe, Porsche Destination Charging, Porsche Smart Mobility, Saascharge, Smart Charge, University of, Wevolt, Zeus Renewables


<a id="t2-type"></a>
## 13. Charger type and status

`Charger_Type` holds two different facts in one column: an electrical type (`AC`, `DC`) and, in 98
records, a lifecycle status (`Upcoming`). That is a modelling problem, not a formatting one. While
they share a column, no query can ask for "all DC chargers" without silently excluding every
planned DC site, and no query can ask "what is planned?" without pattern-matching on a type field.

Splitting them gives `charger_type` ∈ {AC, DC} and `charger_status` ∈ {Operational, Upcoming}. For
upcoming sites the type is genuinely unknown — the source does not record whether a planned site
will be AC or DC — so it is left missing rather than inferred from the power rating. A 350 kW
rating is strong evidence of DC, but *evidence* is not *data*, and Task 3 targets DC chargers
specifically: an inferred DC record would quietly enter that population as though it were observed.

In [6]:
# `Charger_Type` holds two different facts: the electrical type (AC/DC) and, for
# 98 records, a lifecycle status ('Upcoming'). Keeping them in one column means
# no query can ask for "all DC chargers" without silently excluding planned DC
# sites, so the two facts are separated into two fields.
def split_type_status(value):
    if pd.isna(value):
        return (pd.NA, pd.NA)
    text = str(value).strip().upper()
    if text in {"AC", "DC"}:
        return (text, "Operational")
    if text == "UPCOMING":
        # The source does not say whether a planned site will be AC or DC, so the
        # type is left missing rather than guessed from the power rating.
        return (pd.NA, "Upcoming")
    return (pd.NA, "Unknown")


df[["charger_type", "charger_status"]] = pd.DataFrame(
    df["charger_type_raw"].map(split_type_status).tolist(), index=df.index
)

record_step("type/status separated", "'Upcoming' moved into charger_status",
            int((df["charger_status"] == "Upcoming").sum()))
df.groupby(["charger_status", "charger_type"], dropna=False).size().to_frame("records")

  type/status separated          98  'Upcoming' moved into charger_status


records
charger_status charger_type         
Operational    AC               1427
               DC                433
Upcoming       NaN                98

<a id="t2-rating"></a>
## 14. Charger power ratings

`Charger_rating` mixes four formats across 46 distinct strings: `22 kW`, a bare `22`, the
non-numeric placeholder `AC` in 522 rows, and compound values such as `2x350kW & 2x175kW`.

The compound form is the interesting one, and it is why the parser returns a *list* of
`(connectors, kW)` pairs rather than a single number. `2x350kW & 2x175kW` describes two 350 kW
connectors and two 175 kW connectors at one site. Any single-number representation — maximum,
minimum, mean — throws away most of that. The list preserves it, and section 18 unpacks it into a
tidy table that Task 4's schema can store properly.

The placeholder `AC` is treated as missing rather than as a power of zero. It records a charger
*type* in a rating field, so the honest reading is that the rating is unknown.

This cell only defines the parser and profiles the formats. The derived columns come later, after
deduplication, so they describe the surviving record rather than one that was merged away.

In [7]:
RATING_SEPARATOR = re.compile(r"\s*(?:&|\+|,|/|\band\b)\s*", re.IGNORECASE)
RATING_TOKEN = re.compile(
    r"^(?:(?P<count>\d+)\s*x\s*)?(?P<kw>\d+(?:\.\d+)?)\s*(?:kw)?$", re.IGNORECASE
)


def parse_rating(value):
    """
    Turn one raw `Charger_rating` string into a list of (connectors, kW) pairs.

    Covers all four formats present in the source:
        '22 kW'              -> [(1, 22.0)]
        '22'                 -> [(1, 22.0)]
        '2x350kW & 2x175kW'  -> [(2, 350.0), (2, 175.0)]
        'AC'                 -> []            a placeholder, not a power value

    Returning a list rather than one number is the whole point: a compound value
    describes several connectors of different power at one site, and flattening
    it to a single figure destroys information the Task 4 schema needs.
    """
    if pd.isna(value):
        return []
    text = str(value).strip().lower().replace("\u00d7", "x")
    if not text:
        return []
    components = []
    for part in RATING_SEPARATOR.split(text):
        match = RATING_TOKEN.match(part.strip())
        if match:
            count = int(match.group("count")) if match.group("count") else 1
            components.append((count, float(match.group("kw"))))
    return components


def classify_rating(value) -> str:
    """Label which of the four formats a raw rating string uses."""
    text = "" if pd.isna(value) else str(value).strip()
    if re.fullmatch(r"\d+(\.\d+)?\s*kW", text, re.IGNORECASE):
        return "number+unit"
    if re.fullmatch(r"\d+(\.\d+)?", text):
        return "bare number"
    if re.search(r"\dx\s*\d", text, re.IGNORECASE):
        return "compound"
    if re.fullmatch(r"[A-Za-z ]+", text):
        return "non-numeric placeholder"
    return "other"


# Profiling only at this point. The derived numeric columns are computed after
# deduplication, so they are guaranteed to describe the record that survives.
df["charger_rating_raw"].map(classify_rating).value_counts().to_frame("records")

,records
charger_rating_raw,
number+unit,1315
non-numeric placeholder,522
compound,99
bare number,22


<a id="t2-address"></a>
## 15. Addresses and postcodes

The newlines are already gone; what remains is punctuation. Addresses arrive as `, Muswellbrook,
2333` (empty street line), `76 Wingewarra St, Dubbo , 2830` (space before the comma) and
`1 - 7 Ross St, Wilcannia NSW 2836, Australia`. Normalising the comma spacing and stripping the
stray leading and trailing separators gives one consistent single-line form.

The postcode needs a decision rather than a rule, because the dataset carries two of them and they
disagree. `PCODE` is null in 121 records, and demonstrably wrong in others: the Wilcannia record
has `PCODE = 2350` while both its address text and its coordinates place it in 2836. The postcode
embedded in the address is therefore preferred, `PCODE` fills the gaps, and every disagreement is
flagged rather than quietly resolved — the flag is what lets the report quantify how often the two
fields conflict.

The locality is left alone. 399 records give it as the placeholder `Sydney` when the true suburb is
somewhere else entirely, and there is no way to recover the real value from within the data. It is
flagged, and the SA4 join in section 20 supplies a trustworthy geography instead.

In [8]:
POSTCODE_IN_ADDRESS = re.compile(r"\b(\d{4})\b(?!.*\b\d{4}\b)")   # last 4-digit run


def in_nsw_postcode_range(code) -> bool:
    if pd.isna(code) or not str(code).isdigit():
        return False
    number = int(code)
    return any(low <= number <= high for low, high in NSW_POSTCODE_RANGES)


# The newlines are already gone (whitespace pass); what remains is punctuation
# spacing - ' ,', ',,' and stray leading/trailing commas from empty street lines.
df["station_address"] = (
    df["station_address_raw"]
    .str.replace(r"\s*,\s*", ", ", regex=True)
    .str.replace(r"(?:,\s*){2,}", ", ", regex=True)
    .str.replace(r"^[,\s]+", "", regex=True)
    .str.replace(r"[,\s]+$", "", regex=True)
)

df["postcode_from_address"] = df["station_address"].str.extract(POSTCODE_IN_ADDRESS, expand=False)
# The address is the more trustworthy of the two sources: PCODE is null in 121
# records and demonstrably wrong in others (the Wilcannia record carries 2350 but
# its address, and its coordinates, are in 2836).
df["postcode"] = df["postcode_from_address"].fillna(df["postcode_reported"])
df["postcode_conflict_flag"] = (
    df["postcode_from_address"].notna()
    & df["postcode_reported"].notna()
    & (df["postcode_from_address"] != df["postcode_reported"])
)
df["postcode_valid_flag"] = df["postcode"].map(in_nsw_postcode_range)

# The locality inside the address is a known-bad field: 399 records use 'Sydney'
# in place of the real suburb. It is flagged, not repaired - the SA4 join below
# supplies a trustworthy region instead.
df["locality_placeholder_flag"] = df["station_address"].str.contains(
    r",\s*Sydney\s*,", case=False, regex=True, na=False
)

record_step("addresses normalised", "addresses reduced to one clean line",
            int(df["station_address"].notna().sum()))
record_step("postcode recovered", "postcode recovered from the address text",
            int((df["postcode_reported"].isna() & df["postcode"].notna()).sum()))
record_step("postcode conflict", "PCODE disagrees with the address postcode",
            int(df["postcode_conflict_flag"].sum()))
record_step("postcode invalid", "postcode outside the NSW ranges",
            int((~df["postcode_valid_flag"]).sum()))
record_step("locality placeholder", "'Sydney' used in place of the real suburb",
            int(df["locality_placeholder_flag"].sum()))
df[["station_address", "postcode_reported", "postcode", "postcode_conflict_flag"]].head()

  addresses normalised        1,958  addresses reduced to one clean line
  postcode recovered            120  postcode recovered from the address text
  postcode conflict              35  PCODE disagrees with the address postcode
  postcode invalid                3  postcode outside the NSW ranges
  locality placeholder          399  'Sydney' used in place of the real suburb


,station_address,postcode_reported,postcode,postcode_conflict_flag
0,"Muswellbrook, 2333",2333,2333,False
1,"01 Wallgrove Road, Sydney, 2766",2766,2766,False
2,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",2350,2836,True
3,"1 Balfour St, Sydney, 2070",2070,2070,False
4,"1 Bay Ln, Byron Bay, 2481",2481,2481,False


<a id="t2-coords"></a>
## 16. Coordinates and sites

Coordinates were rounded to six decimal places in section 11.3 — about 11 cm, already finer than
the source can justify. That rounding matters here: unrounded floating-point values make two
records at one physical site compare as different locations, which would defeat the deduplication
below.

`site_id` groups records sharing a location. It exists because "duplicate coordinates" is
ambiguous in this dataset. Section 6.5 found 20 records sharing a location with another, but they
are not all duplicates: at 19 Princes Hwy, Figtree, a Tesla DC unit and a non-networked AC unit
genuinely coexist, while the same Tesla charger also appears twice because two feeds describe it
differently. Separating *site* from *charger* lets the next cell treat those two situations
differently instead of applying one rule to both and getting one of them wrong.

In [9]:
missing_coords = df["latitude"].isna() | df["longitude"].isna()
outside_nsw = ~(
    df["latitude"].between(*NSW_LAT_RANGE) & df["longitude"].between(*NSW_LON_RANGE)
)
df["coordinate_valid_flag"] = ~(missing_coords | outside_nsw)

record_step("coordinates missing", "records with no usable coordinates", int(missing_coords.sum()))
record_step("coordinates off-NSW", "records outside the NSW bounding box",
            int((outside_nsw & ~missing_coords).sum()))

# `site_id` groups records that share a physical location. It is what makes the
# difference between a duplicate (same site, same charger, two feeds) and a
# legitimate co-located pair (same site, one AC unit and one DC unit) something
# the data can express, instead of something dedup has to guess at.
site_key = (
    df["latitude"].round(COORDINATE_DECIMALS).astype("string")
    + "," + df["longitude"].round(COORDINATE_DECIMALS).astype("string")
)
df["site_id"] = pd.factorize(site_key)[0] + 1
print(f"{df['site_id'].nunique():,} distinct sites across {len(df):,} records")

  coordinates missing             0  records with no usable coordinates
  coordinates off-NSW             0  records outside the NSW bounding box
1,935 distinct sites across 1,958 records


<a id="t2-dupes"></a>
## 17. Duplicates and reconciliation

Two passes, because there are two problems.

**Exact duplicates** are identical on every field that identifies a charger. One copy is redundant
and is dropped.

**Conflicting duplicates** are the same charger at the same site, described differently by two
source feeds. At 520 David St, Albury, one row reads `AC` / 4 plugs and the other `7` / 2 plugs.
Dropping either loses information the other one has, so the group is merged under rules stated per
field:

| Field | Rule | Why |
|---|---|---|
| rating | prefer a value that parses; then the one describing most connectors | `'7'` beats the placeholder `'AC'`; `'2x350kW & 2x175kW'` beats `'175 kW'` |
| plugs | the larger count | a feed reporting fewer is reporting a subset of the bays |
| text fields | the longest non-null value | the least truncated version of the same string |
| flags | logical OR | a problem noted on either row survives the merge |

The grouping key is `(site_id, operator, charger_type, charger_status)` — and it only works because
operator canonicalisation ran first. Before section 12, the Figtree pair reads `Tesla` and
`Tesla Motors ` and looks like two different chargers.

`charger_id` is assigned last, once the record set is final. It is the primary key in Task 4 and
the join target for Task 3's augmented attributes, so it must be stable — assigning it before
deduplication would leave gaps and shift meaning between runs.

In [10]:
# Two different problems, handled in two passes.
#
# Pass 1 - exact duplicates: identical on every field that identifies a charger.
# One copy is simply redundant.
IDENTITY_COLUMNS = [
    "site_id", "operator", "charger_type", "charger_status",
    "charger_rating_raw", "number_of_plugs", "station_address",
]
exact_duplicates = df.duplicated(subset=IDENTITY_COLUMNS, keep="first")
df = df.loc[~exact_duplicates].copy()
record_step("exact duplicates", "identical records removed", int(exact_duplicates.sum()))

# Pass 2 - conflicting duplicates: the same charger at the same site described
# differently by two feeds (rating 'AC' with 4 plugs in one row, '7' with 2 plugs
# in the other). Dropping either row loses real information, so the group is
# merged under rules chosen per field:
#   rating  - prefer the value that actually parses to a power figure, then the
#             one describing the most connectors; the 'AC' placeholder loses
#   plugs   - the larger count; a feed reporting fewer is reporting a subset
#   text    - the longest non-null value, i.e. the least truncated
#   flags   - OR'd, so a problem noted on either row survives the merge
GROUP_KEY = ["site_id", "operator", "charger_type", "charger_status"]
conflicting = df.duplicated(subset=GROUP_KEY, keep=False)
conflict_groups = int(df.loc[conflicting, GROUP_KEY].drop_duplicates().shape[0])
print(f"{int(conflicting.sum())} record(s) in {conflict_groups} conflicting group(s)")


def longest(series):
    """The longest non-null string in the group - the least truncated one."""
    values = series.dropna().astype(str)
    return max(values, key=len) if len(values) else pd.NA


def best_rating(series):
    """
    The most informative rating in the group.

    Sort key: parses at all > describes more connectors > longer string. This is
    what keeps '7' rather than the placeholder 'AC', and keeps
    '2x350kW & 2x175kW' rather than the single-figure '175 kW'.
    """
    values = series.dropna().astype(str)
    if not len(values):
        return pd.NA
    return max(values, key=lambda v: (len(parse_rating(v)) > 0, len(parse_rating(v)), len(v)))


AGGREGATIONS = {
    "source_object_id": "first",
    "station_name": longest,
    "station_address": longest,
    "station_address_raw": longest,
    "operator_raw": longest,
    "charger_type_raw": "first",
    "charger_rating_raw": best_rating,
    "number_of_plugs": "max",
    "latitude": "first",
    "longitude": "first",
    "lga_name": longest,
    "postcode": longest,
    "postcode_reported": longest,
    "postcode_from_address": longest,
    "source_feed": longest,
    "operator_truncated_flag": "max",
    "postcode_conflict_flag": "max",
    "postcode_valid_flag": "max",
    "locality_placeholder_flag": "max",
    "coordinate_valid_flag": "min",
}

before_rows = len(df)
df = df.groupby(GROUP_KEY, dropna=False, as_index=False).agg(AGGREGATIONS)
record_step("duplicates reconciled", "conflicting records merged into one",
            before_rows - len(df))

# A stable surrogate key, assigned once the record set is final. Task 4 uses it
# as the primary key; Task 3 uses it as the join target for augmented attributes.
df = df.sort_values(["site_id", "operator", "charger_type"]).reset_index(drop=True)
df.insert(0, "charger_id", range(1, len(df) + 1))
print(f"{before_rows:,} -> {len(df):,} records after reconciliation")

  exact duplicates                0  identical records removed
24 record(s) in 12 conflicting group(s)
  duplicates reconciled          12  conflicting records merged into one
1,958 -> 1,946 records after reconciliation


<a id="t2-ratingfields"></a>
## 18. Derived rating fields

Now that each charger is one record, the parser from section 14 is applied to produce
`power_kw_min`, `power_kw_max` and a tidy component table.

One subtlety in `connectors_in_rating`: only a compound rating states how many connectors exist.
`22 kW` states the power *per* connector and says nothing at all about their number, so counting it
as one connector would invent a fact — and would make the plug-count cross-check in section 19 fire
on almost every row, drowning the real signal in noise. The column is therefore left missing except
where the rating explicitly counts connectors.

The component table is what makes the compound ratings queryable. `power_kw_max` answers "how fast
is this charger?"; `charger_power_ratings.csv` answers "what is actually installed there?" — a
question a single column cannot hold an answer to, and one that Task 4's normalised schema is
designed for.

In [11]:
# The derived numeric fields are computed here, after reconciliation, so they
# always describe the rating string that actually survived the merge.
parsed = df["charger_rating_raw"].map(parse_rating)
df["rating_format"] = df["charger_rating_raw"].map(classify_rating)
df["power_kw_max"] = parsed.map(lambda comps: max(kw for _, kw in comps) if comps else np.nan)
df["power_kw_min"] = parsed.map(lambda comps: min(kw for _, kw in comps) if comps else np.nan)

# Only a compound rating ('2x350kW') states how many connectors exist. A plain
# '22 kW' states the power *per* connector and says nothing about their number,
# so treating it as one connector would invent a fact and make the plug-count
# cross-check below fire on almost every row.
df["connectors_in_rating"] = (
    parsed.map(lambda comps: sum(count for count, _ in comps) if comps else pd.NA)
          .where(df["rating_format"] == "compound", pd.NA)
          .astype("Int64")
)

record_step("rating parsed", "ratings resolved to a numeric kW value",
            int(df["power_kw_max"].notna().sum()))
record_step("rating unparsable", "placeholder ratings left as NA (e.g. 'AC')",
            int(df["power_kw_max"].isna().sum()))

# The compound ratings are also emitted as a tidy one-row-per-component table.
# `power_kw_max` answers "how fast is this charger?"; this answers "what is
# actually installed there?", which is what a normalised schema needs and what a
# single column cannot hold.
power_ratings = pd.DataFrame(
    [
        {"charger_id": charger_id, "component_no": position,
         "connector_count": count, "power_kw": kw}
        for charger_id, components in zip(df["charger_id"], parsed)
        for position, (count, kw) in enumerate(components, start=1)
    ],
    columns=["charger_id", "component_no", "connector_count", "power_kw"],
)
print(f"{len(power_ratings):,} rating components for "
      f"{power_ratings['charger_id'].nunique():,} chargers")
df["rating_format"].value_counts().to_frame("records")

  rating parsed               1,428  ratings resolved to a numeric kW value
  rating unparsable             518  placeholder ratings left as NA (e.g. 'AC')
1,527 rating components for 1,428 chargers


,records
rating_format,
number+unit,1314
non-numeric placeholder,518
compound,99
bare number,15


<a id="t2-consistency"></a>
## 19. Cross-field consistency

These checks change no values. They mark records whose own fields contradict each other, which is
the "inconsistent charger attributes" the brief asks about — inconsistency *within* a record, not
just between records.

Three checks run. A `DC` charger whose rating reads `AC` cannot be both. A compound rating counting
four connectors on a record reporting six plugs disagrees with itself. And a missing station name,
while not a contradiction, matters enough to Task 3 to be worth a flag: with 432 of 433 DC chargers
unnamed, name-based matching against an external API is not viable and coordinates are the only
usable join key.

In [12]:
# Cross-field checks. These change no values; they mark records whose own fields
# contradict each other, so the report can quantify consistency and Task 3 can
# avoid augmenting a record that is internally unreliable.
df["type_rating_conflict_flag"] = (
    (df["charger_type"] == "DC")
    & df["charger_rating_raw"].str.upper().eq("AC").fillna(False)
)
df["plug_count_conflict_flag"] = (
    df["connectors_in_rating"].notna()
    & df["number_of_plugs"].notna()
    & (df["connectors_in_rating"] != df["number_of_plugs"])
)
df["missing_name_flag"] = df["station_name"].isna()

record_step("type vs rating", "DC records whose rating reads 'AC'",
            int(df["type_rating_conflict_flag"].sum()))
record_step("plugs vs connectors", "plug count disagrees with the parsed rating",
            int(df["plug_count_conflict_flag"].sum()))
record_step("station name missing", "records with no station name",
            int(df["missing_name_flag"].sum()))

  type vs rating                  0  DC records whose rating reads 'AC'
  plugs vs connectors            36  plug count disagrees with the parsed rating
  station name missing        1,430  records with no station name


<a id="t2-spatial"></a>
## 20. Spatial integration with ASGS SA4

This is the join the brief asks for: add a field to each EV charger naming the SA4 region it falls
in. It runs in DuckDB with the `spatial` extension, for the reason given in section 7 — Task 4 has
to store this data in DuckDB anyway, so doing the geometry there keeps one engine rather than
moving data between GeoPandas and a database.

### 20.1 Reconciling the coordinate reference systems

The two datasets do not share a CRS. The ABS boundaries are GDA2020 (EPSG:7844); the TfNSW
latitude and longitude columns are WGS84 (EPSG:4326). At the current epoch the two differ by under
about 2 m, which cannot move a charger across an SA4 boundary that is kilometres wide — but
"close enough" is not a decision worth leaving implicit in a spatial join, so the polygons are
reprojected explicitly and the choice is recorded in the ledger.

`always_xy := true` is the part that actually bites. EPSG:4326 formally orders its axes
latitude-then-longitude, while `ST_Point` takes `(x, y)` — longitude first. Without that flag the
transform returns coordinates in the opposite order from the points being tested against them, and
every charger silently matches nothing.

In [13]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

# The boundary file is GDA2020 (EPSG:7844); the TfNSW coordinates are WGS84
# (EPSG:4326). The two are close - under about 2 m at the current epoch - but
# "close enough" is not a decision worth leaving implicit in a spatial join, so
# the polygons are reprojected explicitly.
#
# `always_xy := true` is the part that matters. EPSG:4326 formally orders its
# axes latitude-then-longitude, so without it the transform returns coordinates
# in the opposite order from ST_Point(longitude, latitude) and every point lands
# in the ocean off East Africa. If the local PROJ build cannot perform the
# transform, the untransformed geometry is used and the fallback is logged: a
# sub-2 m offset cannot move a charger across an SA4 boundary kilometres wide.
TRANSFORMED = f"ST_Transform(geom, '{SOURCE_CRS}', '{TARGET_CRS}', always_xy := true)"
SA4_COLUMNS = """
    SA4_CODE26 AS sa4_code,
    SA4_NAME26 AS sa4_name,
    GCC_CODE26 AS gcc_code,
    GCC_NAME26 AS gcc_name,
    STE_CODE26 AS state_code,
    STE_NAME26 AS state_name,
    AREASQKM26 AS area_sqkm
"""
try:
    con.execute(f"""
        CREATE OR REPLACE TABLE sa4 AS
        SELECT {SA4_COLUMNS}, {TRANSFORMED} AS geom
        FROM ST_Read('{SA4_SHAPEFILE_PATH.as_posix()}')
    """)
    crs_note = f"reprojected {SOURCE_CRS} -> {TARGET_CRS} (always_xy)"
except duckdb.Error as error:
    print(f"WARNING: ST_Transform unavailable ({str(error)[:120]}); "
          "using GDA2020 coordinates directly (offset < 2 m).")
    con.execute(f"""
        CREATE OR REPLACE TABLE sa4 AS
        SELECT {SA4_COLUMNS}, geom
        FROM ST_Read('{SA4_SHAPEFILE_PATH.as_posix()}')
    """)
    crs_note = f"{SOURCE_CRS} treated as {TARGET_CRS} (offset < 2 m)"

record_step("CRS reconciled", crs_note,
            int(con.execute("SELECT COUNT(*) FROM sa4").fetchone()[0]))
con.execute("SELECT state_name, COUNT(*) AS regions FROM sa4 GROUP BY 1 ORDER BY 2 DESC").df()

  CRS reconciled                108  reprojected EPSG:7844 -> EPSG:4326 (always_xy)


,state_name,regions
0,New South Wales,30
1,Queensland,21
2,Victoria,19
3,Western Australia,12
4,South Australia,9
5,Tasmania,6
6,Northern Territory,4
7,Other Territories,3
8,Australian Capital Territory,3
9,Outside Australia,1


### 20.2 The point-in-polygon join

Each charger becomes a point and is tested for containment in each SA4 polygon.

The join runs against all 108 Australian SA4s rather than a pre-filtered NSW subset, which is a
deliberate choice. Filtering to NSW first would make an interstate charger — a genuine data error
worth reporting — come back unmatched and indistinguishable from a geometry failure. Joining
against the whole country means the result *tells* us which state each point is in, and section 21
asserts that every one of them is NSW.

A `LEFT JOIN` rather than an inner join, so that an unmatched charger survives as a row with a null
region instead of vanishing from the dataset.

In [14]:
# The cleaned chargers are handed to DuckDB as a view over the DataFrame - no
# copy, no intermediate file.
chargers_for_join = df[["charger_id", "longitude", "latitude"]].astype(
    {"charger_id": "int64", "longitude": "float64", "latitude": "float64"}
)
con.register("chargers_py", chargers_for_join)

# ST_Point takes (x, y) = (longitude, latitude). Passing them the other way round
# is the most common error in this kind of join and it fails silently - every
# point simply matches nothing.
#
# The join runs against all 108 Australian SA4s rather than a pre-filtered NSW
# subset. Filtering first would hide a genuine problem: a charger whose
# coordinates put it outside NSW would come back unmatched and look like a
# geometry failure, instead of being correctly reported as an interstate point.
con.execute("""
    CREATE OR REPLACE TABLE charger_sa4 AS
    SELECT c.charger_id,
           s.sa4_code, s.sa4_name, s.gcc_code, s.gcc_name, s.state_name,
           CASE WHEN s.sa4_code IS NULL THEN NULL ELSE 'point-in-polygon' END
               AS sa4_match_method
    FROM chargers_py AS c
    LEFT JOIN sa4 AS s
      ON ST_Contains(s.geom, ST_Point(c.longitude, c.latitude))
""")

matched = con.execute(
    "SELECT COUNT(*) FROM charger_sa4 WHERE sa4_code IS NOT NULL"
).fetchone()[0]
print(f"{matched:,} of {len(df):,} chargers matched by point-in-polygon "
      f"({matched / len(df) * 100:.1f}%)")

1,945 of 1,946 chargers matched by point-in-polygon (99.9%)


### 20.3 Chargers that fall outside every polygon

A charger on a jetty, a reclaimed wharf or a coastal car park can sit a few metres outside the
coastline the ABS digitised, and point-in-polygon returns nothing for it. Dropping those records
would bias the coverage analysis toward inland regions; leaving the field null would push the
problem into Task 4.

Instead each unmatched charger is assigned the nearest SA4, but only within a stated tolerance of
about 2 km, and `sa4_match_method` records which chargers were assigned this way. That last part is
what keeps the fallback honest: the report can state exactly how many regions were assigned exactly
and how many were assisted, rather than presenting both as the same kind of result.

`ST_Distance` on geographic coordinates returns degrees, not metres, which is why the tolerance is
expressed in degrees. At NSW latitudes 0.02° is roughly 2 km — precise enough for a sanity
threshold, and the exact figure does not matter as long as it is small.

In [15]:
# A charger on a jetty, a reclaimed wharf or a coastal car park can sit a few
# metres outside the coastline the ABS digitised, and point-in-polygon returns
# nothing for it. Rather than dropping those records or leaving the field null,
# each is assigned the nearest SA4 - but only within a stated tolerance, and the
# method is recorded in `sa4_match_method` so the report can separate exact
# matches from assisted ones.
NEAREST_TOLERANCE_DEGREES = 0.02       # ~2 km at NSW latitudes

unmatched_count = con.execute(
    "SELECT COUNT(*) FROM charger_sa4 WHERE sa4_code IS NULL"
).fetchone()[0]

if unmatched_count:
    con.execute(f"""
        CREATE OR REPLACE TABLE charger_sa4_nearest AS
        WITH candidates AS (
            SELECT u.charger_id, s.sa4_code, s.sa4_name, s.gcc_code, s.gcc_name,
                   s.state_name,
                   ST_Distance(s.geom, ST_Point(u.longitude, u.latitude)) AS distance_deg,
                   ROW_NUMBER() OVER (
                       PARTITION BY u.charger_id
                       ORDER BY ST_Distance(s.geom, ST_Point(u.longitude, u.latitude))
                   ) AS rank
            FROM chargers_py AS u
            JOIN charger_sa4 AS m
              ON m.charger_id = u.charger_id AND m.sa4_code IS NULL
            CROSS JOIN sa4 AS s
            WHERE NOT ST_IsEmpty(s.geom)
        )
        SELECT charger_id, sa4_code, sa4_name, gcc_code, gcc_name,
               state_name, distance_deg
        FROM candidates
        WHERE rank = 1 AND distance_deg <= {NEAREST_TOLERANCE_DEGREES}
    """)
    con.execute("""
        UPDATE charger_sa4 AS t
        SET sa4_code = n.sa4_code, sa4_name = n.sa4_name,
            gcc_code = n.gcc_code, gcc_name = n.gcc_name,
            state_name = n.state_name, sa4_match_method = 'nearest-polygon'
        FROM charger_sa4_nearest AS n
        WHERE t.charger_id = n.charger_id
    """)
    rescued = con.execute(
        "SELECT COUNT(*) FROM charger_sa4 WHERE sa4_match_method = 'nearest-polygon'"
    ).fetchone()[0]
    print(f"{rescued} of {unmatched_count} unmatched charger(s) assigned by nearest polygon")
else:
    print("Every charger fell inside an SA4 polygon; no fallback needed.")

still_unmatched = con.execute(
    "SELECT COUNT(*) FROM charger_sa4 WHERE sa4_code IS NULL"
).fetchone()[0]
record_step("SA4 assigned", "chargers given an SA4 region", len(df) - still_unmatched)
record_step("SA4 unassigned", "chargers left without an SA4 region", still_unmatched)

1 of 1 unmatched charger(s) assigned by nearest polygon
  SA4 assigned                1,946  chargers given an SA4 region
  SA4 unassigned                  0  chargers left without an SA4 region


### 20.4 Merging the regions back

The SA4 fields join back onto the cleaned DataFrame on `charger_id`. `validate="one_to_one"` makes
pandas raise if the join is not one-to-one, catching a duplicated key immediately rather than
letting it silently inflate the row count.

The NSW SA4 table is also kept as an output in its own right. It becomes the region dimension in
Task 4, and it is the only thing that can show a region with **no** chargers at all — a join result
can only ever list regions that already have one, and for a coverage analysis the empty regions are
the interesting ones.

In [16]:
sa4_lookup = con.execute("""
    SELECT charger_id, sa4_code, sa4_name, gcc_code, gcc_name,
           state_name, sa4_match_method
    FROM charger_sa4
""").df()

df = df.merge(sa4_lookup, on="charger_id", how="left", validate="one_to_one")

# The NSW SA4 reference table becomes its own output: Task 4 stores it as the
# region dimension, and it is what makes a region with *no* chargers visible.
# A join result alone can only ever show regions that already have one.
sa4_regions_nsw = con.execute("""
    SELECT sa4_code, sa4_name, gcc_code, gcc_name, state_name,
           ROUND(area_sqkm, 1) AS area_sqkm
    FROM sa4
    WHERE state_name = 'New South Wales'
    ORDER BY sa4_code
""").df()

print(f"{len(sa4_regions_nsw)} NSW SA4 regions; "
      f"{df['sa4_code'].nunique()} of them contain at least one charger")
df[["charger_id", "operator", "charger_type", "latitude", "longitude",
    "sa4_code", "sa4_name", "sa4_match_method"]].head()

30 NSW SA4 regions; 28 of them contain at least one charger


,charger_id,operator,charger_type,latitude,longitude,sa4_code,sa4_name,sa4_match_method
0,1,EVUp,AC,-32.262242,150.890139,106,Hunter Valley exc Newcastle,point-in-polygon
1,2,BP Australia,DC,-33.811004,150.849597,116,Sydney - Blacktown,point-in-polygon
2,3,NRMA,DC,-30.511874,151.669395,110,New England and North West,point-in-polygon
3,4,Chargefox,AC,-33.774101,151.167035,121,Sydney - North Sydney and Hornsby,point-in-polygon
4,5,Tesla,AC,-28.641819,153.613633,112,Richmond - Tweed,point-in-polygon


<a id="t2-validate"></a>
## 21. Validation and summary

Each check below is an assumption the rest of the pipeline relies on: that `charger_id` is unique,
that `charger_type` holds nothing but AC and DC, that every referenced charger in the ratings table
exists, that no charger landed in another state. Stating them as checks rather than eyeballing the
output means a future TfNSW release that breaks one stops the notebook here, instead of surfacing
as a wrong number in a Task 4 query.

In [17]:
# Post-conditions. Each is an assumption the rest of the pipeline relies on, so
# each is asserted rather than eyeballed: a future TfNSW release that breaks one
# should stop the notebook here, not surface as a wrong number in Task 4.
problems = []

if df["charger_id"].duplicated().any():
    problems.append("charger_id is not unique")
if df["charger_id"].isna().any():
    problems.append("charger_id contains nulls")
if not set(df["charger_type"].dropna()) <= {"AC", "DC"}:
    problems.append(f"unexpected charger_type values: {set(df['charger_type'].dropna())}")
if not set(df["charger_status"].dropna()) <= {"Operational", "Upcoming", "Unknown"}:
    problems.append("unexpected charger_status values")
if not bool(df["latitude"].dropna().between(*NSW_LAT_RANGE).all()):
    problems.append("latitude outside the NSW range")
if not bool(power_ratings["charger_id"].isin(df["charger_id"]).all()):
    problems.append("power_ratings references an unknown charger_id")

off_state = df.loc[df["state_name"].notna() & (df["state_name"] != "New South Wales")]
if len(off_state):
    problems.append(f"{len(off_state)} charger(s) matched an SA4 outside NSW")

print("Validation:", "PASSED" if not problems else "FAILED")
for problem in problems:
    print("  -", problem)

summary = pd.DataFrame({
    "metric": [
        "raw records", "clean records", "distinct sites", "distinct operators",
        "DC chargers", "AC chargers", "upcoming sites",
        "records with an SA4", "NSW SA4 regions covered",
        "NSW SA4 regions with no charger",
    ],
    "value": [
        len(raw), len(df), df["site_id"].nunique(), df["operator"].nunique(),
        int((df["charger_type"] == "DC").sum()), int((df["charger_type"] == "AC").sum()),
        int((df["charger_status"] == "Upcoming").sum()),
        int(df["sa4_code"].notna().sum()), int(df["sa4_code"].nunique()),
        int(len(sa4_regions_nsw) - df["sa4_code"].nunique()),
    ],
})
summary

Validation: PASSED


,metric,value
0,raw records,1958
1,clean records,1946
2,distinct sites,1935
3,distinct operators,42
4,DC chargers,431
5,AC chargers,1417
6,upcoming sites,98
7,records with an SA4,1946
8,NSW SA4 regions covered,28
9,NSW SA4 regions with no charger,2


The per-region distribution below is the first thing the cleaned data makes possible, and it is
worth including in the report: it is built from the full SA4 list rather than from the join result,
so regions with zero chargers appear as zeros instead of disappearing.

In [18]:
chargers_by_sa4 = (
    sa4_regions_nsw[["sa4_code", "sa4_name", "gcc_name"]]
    .merge(
        df.groupby("sa4_code")
          .agg(chargers=("charger_id", "count"),
               dc_chargers=("charger_type", lambda s: int((s == "DC").sum())),
               plugs=("number_of_plugs", "sum"))
          .reset_index(),
        on="sa4_code", how="left",
    )
    .fillna({"chargers": 0, "dc_chargers": 0, "plugs": 0})
    .astype({"chargers": int, "dc_chargers": int, "plugs": int})
    .sort_values("chargers", ascending=False)
    .reset_index(drop=True)
)
chargers_by_sa4

,sa4_code,sa4_name,gcc_name,chargers,dc_chargers,plugs
0,118,Sydney - Eastern Suburbs,Greater Sydney,219,34,399
1,117,Sydney - City and Inner South,Greater Sydney,139,19,388
2,106,Hunter Valley exc Newcastle,Rest of NSW,127,9,361
3,101,Capital Region,Rest of NSW,117,27,411
4,103,Central West,Rest of NSW,113,17,269
5,120,Sydney - Inner West,Greater Sydney,113,17,238
6,121,Sydney - North Sydney and Hornsby,Greater Sydney,108,41,346
7,111,Newcastle and Lake Macquarie,Rest of NSW,86,13,250
8,112,Richmond - Tweed,Rest of NSW,76,14,213
9,114,Southern Highlands and Shoalhaven,Rest of NSW,76,9,203


<a id="t2-outputs"></a>
## 22. Outputs

Three datasets and the cleaning ledger are written to `data/interim/`. Task 3 reads the charger
file to augment the DC records; Task 4 reads all three to populate the DuckDB schema.

The cleaned file keeps the raw values it was derived from — `operator_raw`, `charger_rating_raw`,
`station_address_raw` — alongside the cleaned ones. That is deliberate: a cleaning decision that
cannot be checked against what it replaced cannot really be reviewed, and a marker or teammate
should be able to see both without re-running the pipeline.

In [19]:
COLUMN_ORDER = [
    "charger_id", "site_id",
    "station_name", "station_address", "lga_name", "postcode",
    "latitude", "longitude",
    "sa4_code", "sa4_name", "gcc_code", "gcc_name", "state_name", "sa4_match_method",
    "operator", "charger_type", "charger_status",
    "number_of_plugs", "power_kw_min", "power_kw_max", "connectors_in_rating",
    "rating_format", "charger_rating_raw", "operator_raw", "charger_type_raw",
    "source_feed", "source_object_id", "postcode_reported", "station_address_raw",
    "operator_truncated_flag", "postcode_conflict_flag", "postcode_valid_flag",
    "locality_placeholder_flag", "coordinate_valid_flag",
    "type_rating_conflict_flag", "plug_count_conflict_flag", "missing_name_flag",
]
chargers_clean = df[[column for column in COLUMN_ORDER if column in df.columns]].copy()

chargers_clean.to_csv(CLEAN_CHARGERS_PATH, index=False)
power_ratings.to_csv(RATINGS_PATH, index=False)
sa4_regions_nsw.to_csv(SA4_NSW_PATH, index=False)
CLEANING_LOG_PATH.write_text(
    json.dumps(
        {"raw_records": len(raw), "clean_records": len(chargers_clean),
         "steps": cleaning_log},
        indent=2,
    ) + "\n",
    encoding="utf-8",
)

for path in (CLEAN_CHARGERS_PATH, RATINGS_PATH, SA4_NSW_PATH, CLEANING_LOG_PATH):
    print(f"  {path.relative_to(PROJECT_ROOT)}  ({path.stat().st_size / 1024:,.1f} KB)")

con.close()
pd.DataFrame(cleaning_log)

  data/interim/ev_chargers_clean.csv  (695.4 KB)
  data/interim/charger_power_ratings.csv  (20.0 KB)
  data/interim/sa4_regions_nsw.csv  (2.1 KB)
  data/interim/cleaning_log.json  (2.9 KB)


,step,detail,records_affected
0,whitespace normalised,cells whose text was rewritten,808
1,empty -> NA,empty strings converted to missing values,3638
2,operator canonicalised,values rewritten to a canonical name,181
3,operator truncated,unresolved truncations flagged for review,1
4,type/status separated,'Upcoming' moved into charger_status,98
5,addresses normalised,addresses reduced to one clean line,1958
6,postcode recovered,postcode recovered from the address text,120
7,postcode conflict,PCODE disagrees with the address postcode,35
8,postcode invalid,postcode outside the NSW ranges,3
9,locality placeholder,'Sydney' used in place of the real suburb,399


### 22.1 What Task 2 changed

| Issue found in Task 1 | How Task 2 handled it |
|---|---|
| Embedded newlines in 733 addresses | Collapsed to single-line form before any string comparison |
| 50 operator strings for ~35 entities | Explicit lookup table; unresolvable truncations flagged, not guessed |
| `Charger_rating` mixing 4 formats | Parsed to `power_kw_min` / `power_kw_max` plus a per-connector table |
| `Upcoming` stored as a charger type | Split into `charger_type` and `charger_status` |
| `PCODE` null in 121 rows, wrong in others | Postcode taken from the address text; conflicts flagged |
| Locality recorded as the placeholder `Sydney` | Flagged, not repaired; SA4 supplies the reliable geography |
| Duplicate coordinates | Two-pass dedup: exact removal, then rule-based reconciliation |
| CRS mismatch (GDA2020 vs WGS84) | Reprojected explicitly with `always_xy` before the join |

### 22.2 What remains open, and why

Some things are recorded rather than fixed, and the report should say so plainly:

* **Truncated operator names with no external reference.** `University of` could be any of several
  institutions. It is flagged; resolving it needs the operator lookup that Task 3 builds.
* **The 522 `AC` placeholder ratings.** No power figure exists in the source for these. Task 3's
  augmentation from Open Charge Map or operator sites is the natural place to recover them.
* **Placeholder localities.** Only reverse geocoding could repair the suburb, and the SA4 field now
  serves the purpose the locality would have served, so it is not worth the API budget.
* **Plug counts that disagree with compound ratings.** Both figures come from the same source with
  no tiebreaker; the disagreement is flagged so a query can exclude those records rather than
  averaging away a real contradiction.

Task 3 continues from `data/interim/ev_chargers_clean.csv`, augmenting the DC records.